In [ ]:
# Import libraries

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# Load, clean, and merge meteorological and generation data

import pandas as pd
import numpy as np
import os

MOTI_PATH = "sample_file1.csv"
GEN_PATH  = "sample_file2.csv"

assert os.path.exists(MOTI_PATH), f"Input file not found: {MOTI_PATH}"
assert os.path.exists(GEN_PATH),  f"Input file not found: {GEN_PATH}"

moti_df = pd.read_csv(MOTI_PATH)
generated_df = pd.read_csv(GEN_PATH)

def parse_dt_series(s, dayfirst=None):
    dt = pd.to_datetime(s, errors="coerce", infer_datetime_format=True, dayfirst=dayfirst)
    return dt

assert "datetime" in moti_df.columns, "sample_file1.csv must contain the 'datetime' column"
moti_df["datetime"] = parse_dt_series(moti_df["datetime"], dayfirst=None)

if "Date" in generated_df.columns:
    generated_df["datetime"] = parse_dt_series(generated_df["Date"], dayfirst=True)
    generated_df = generated_df.drop(columns=["Date"])
else:
    assert "datetime" in generated_df.columns, "sample_file2.csv must contain either a 'Date' or 'datetime' column"
    generated_df["datetime"] = parse_dt_series(generated_df["datetime"], dayfirst=None)

columns_to_keep = [
    "datetime", "tempmax", "temp", "humidity", "cloudcover",
    "solarenergy", "precipcover", "sunrise", "sunset", "windspeed", "conditions"
]
missing_cols = [c for c in columns_to_keep if c not in moti_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in sample_file1.csv: {missing_cols}")

moti_df = moti_df[columns_to_keep].copy()

num_cols = ["tempmax","temp","humidity","cloudcover","solarenergy","precipcover","windspeed"]
for c in num_cols:
    moti_df[c] = pd.to_numeric(moti_df[c], errors="coerce")

moti_df = moti_df.dropna(subset=["datetime", "solarenergy"]).copy()

moti_df["DOY"] = moti_df["datetime"].dt.dayofyear
moti_df["DOY_sin"] = np.sin(2*np.pi*moti_df["DOY"]/365.0)
moti_df["DOY_cos"] = np.cos(2*np.pi*moti_df["DOY"]/365.0)
moti_df["humidity_wind"] = moti_df["humidity"] * moti_df["windspeed"]

def to_time_seconds(x):
    if pd.isna(x):
        return np.nan
    dt = pd.to_datetime(x, errors="coerce", infer_datetime_format=True)
    if pd.notna(dt):
        return dt.hour*3600 + dt.minute*60 + dt.second
    try:
        t = pd.to_datetime(str(x), format="%H:%M:%S", errors="coerce")
        if pd.notna(t):
            return t.hour*3600 + t.minute*60 + t.second
    except Exception:
        pass
    return np.nan

sunrise_sec = moti_df["sunrise"].apply(to_time_seconds)
sunset_sec  = moti_df["sunset"].apply(to_time_seconds)

moti_df["Day_Length"] = (sunset_sec - sunrise_sec) / 3600.0
moti_df = moti_df.drop(columns=["sunrise","sunset"]).copy()

# Keep weather conditions as categorical features and one-hot encode them later.
# Ordinal encoding is not used because weather conditions have no natural order.

generated_df = generated_df.copy()

if "Generated" not in generated_df.columns:
    
    raise ValueError("sample_file2.csv must contain the 'Generated' column")
generated_df["Generated"] = pd.to_numeric(generated_df["Generated"], errors="coerce")

merged_df = pd.merge(moti_df, generated_df, on="datetime", how="inner")  # inner join for stability
merged_df = merged_df.dropna().copy()

merged_df = merged_df[merged_df["Day_Length"] > 0].copy()

merged_df = merged_df.sort_values("datetime").reset_index(drop=True)

merged_df.to_csv("Dataset_PINN_Input.csv", index=False)
print(f"✅ Dataset_PINN_Input.csv saved with {len(merged_df)} rows and {len(merged_df.columns)} columns.")


In [ ]:
# Inspect raw dataset before quality filtering

import pandas as pd
import numpy as np

df = pd.read_csv("Dataset_PINN_Input.csv", parse_dates=["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)

print("Duplicates by date:", df['datetime'].duplicated().sum())

print("Is datetime strictly increasing?", df['datetime'].is_monotonic_increasing)

print("solarenergy < 0:", (df['solarenergy'] < 0).sum())
print("Generated < 0:", (df['Generated'] < 0).sum())

q1, q3 = np.percentile(df['Generated'], [25, 75])
iqr = q3 - q1
lb, ub = q1 - 1.5*iqr, q3 + 1.5*iqr
print("Generated outliers (IQR):", ((df['Generated'] < lb) | (df['Generated'] > ub)).sum())

if 'Day_Length' in df.columns:
    print("Day_Length <= 0:", (df['Day_Length'] <= 0).sum())


In [ ]:
# Train-calibrated quality-control filter

import pandas as pd
import numpy as np

PATH = "Dataset_PINN_Input.csv"
OUT_PATH = "Dataset_PINN_Input_filtered.csv"

df0 = pd.read_csv(PATH, parse_dates=["datetime"]).sort_values("datetime").reset_index(drop=True)

num_cols = [
    "Generated","solarenergy","tempmax","temp","humidity","cloudcover",
    "windspeed","precipcover","Day_Length"
]
for c in num_cols:
    if c in df0.columns:
        df0[c] = pd.to_numeric(df0[c], errors="coerce")

report = {}
mask_keep = pd.Series(True, index=df0.index, dtype=bool)

def apply_rule(mask_ok, name):
    mask_ok = pd.Series(mask_ok, index=df0.index).fillna(False).astype(bool)
    removed = int(((~mask_ok) & mask_keep).sum())
    report[name] = removed
    return mask_keep & mask_ok

key_cols = [
    "datetime","Generated","solarenergy","tempmax","humidity",
    "cloudcover","windspeed","precipcover","Day_Length"
]
mask_nonan = df0[key_cols].notna().all(axis=1)
mask_keep = apply_rule(mask_nonan, "missing_core_fields")

phys = pd.Series(True, index=df0.index, dtype=bool)
phys &= (df0["Generated"] >= 0)
phys &= (df0["solarenergy"] >= 0)
phys &= df0["humidity"].between(0, 100, inclusive="both")
phys &= df0["cloudcover"].between(0, 100, inclusive="both")
phys &= df0["precipcover"].between(0, 100, inclusive="both")
phys &= df0["tempmax"].between(-30, 60, inclusive="both")
phys &= df0["windspeed"].between(0, 216, inclusive="both")
phys &= df0["Day_Length"].between(0, 24, inclusive="both")
mask_keep = apply_rule(phys, "hard_physical_bounds")

no_dupes = ~df0.duplicated(subset=["datetime"], keep="first")
mask_keep = apply_rule(no_dupes, "duplicate_datetime")

num_panels = 42
panel_area = 1.3
A = num_panels * panel_area

eta_ref = 0.185
beta = 0.004
tilt_deg = 1.0
cos_theta = float(np.cos(np.radians(tilt_deg)))

sel_idx = mask_keep[mask_keep].index
T = df0.loc[sel_idx, "tempmax"].astype(float).to_numpy()
H = df0.loc[sel_idx, "solarenergy"].astype(float).to_numpy()

eta = eta_ref * (1 - beta * (T - 25.0))
eta = np.clip(eta, 0.05, 0.25)

E_phys = (H * A * eta * cos_theta).astype(float)  # kWh/day proxy
df0.loc[sel_idx, "E_phys_tmp_kWh"] = E_phys

proxy_idx = df0.loc[mask_keep, "datetime"].sort_values().index
n_proxy = len(proxy_idx)
assert n_proxy > 100, "Dataset is too small after basic QC for train-calibrated filtering."

proxy_train_idx = proxy_idx[: int(0.75 * n_proxy)]

EPS = 1e-6
MIN_EPHYS = 3.0

H_train = df0.loc[proxy_train_idx, "solarenergy"].astype(float)
G_train = df0.loc[proxy_train_idx, "Generated"].astype(float)
E_train = df0.loc[proxy_train_idx, "E_phys_tmp_kWh"].astype(float)

ratio_train = G_train.to_numpy() / np.maximum(E_train.to_numpy(), EPS)
ratio_valid_train = (E_train.to_numpy() >= MIN_EPHYS)

if ratio_valid_train.sum() > 50:
    r99 = float(np.nanpercentile(ratio_train[ratio_valid_train], 99))
    OVER_FACTOR = float(max(1.50, min(2.20, r99 + 0.10)))
else:
    OVER_FACTOR = 1.70

H_HIGH_Q = 0.85
Y_LOW_Q  = 0.08
E_HIGH_Q = 0.85

H_high = float(H_train.quantile(H_HIGH_Q)) if len(H_train) else 0.0
Y_low  = float(G_train.quantile(Y_LOW_Q)) if len(G_train) else 0.0
E_high = float(E_train.quantile(E_HIGH_Q)) if len(E_train) else 0.0

Y_LOW_ABS = 2.0
Y_low_thr = max(Y_low, Y_LOW_ABS)

H_MIN_NIGHT = 0.30
Y_MIN_NIGHT = 2.0

sel_idx = mask_keep[mask_keep].index
gen_all = df0.loc[sel_idx, "Generated"].astype(float).to_numpy()
ephys_all = df0.loc[sel_idx, "E_phys_tmp_kWh"].astype(float).to_numpy()

ratio_all = gen_all / np.maximum(ephys_all, EPS)
ratio_valid_all = (ephys_all >= MIN_EPHYS)

ok_overprod = np.ones_like(ephys_all, dtype=bool)
ok_overprod[ratio_valid_all] = (ratio_all[ratio_valid_all] <= OVER_FACTOR)

phys_consistent = pd.Series(True, index=df0.index, dtype=bool)
phys_consistent.loc[sel_idx] = ok_overprod
mask_keep = apply_rule(phys_consistent, "physics_overproduction_only")

# Quality-control thresholds are calibrated on the training period only.
sel_idx = mask_keep[mask_keep].index
H_sel = df0.loc[sel_idx, "solarenergy"].astype(float)
G_sel = df0.loc[sel_idx, "Generated"].astype(float)
E_sel = df0.loc[sel_idx, "E_phys_tmp_kWh"].astype(float)

ok_night = ~((H_sel < H_MIN_NIGHT) & (G_sel > Y_MIN_NIGHT))
ok_outage = ~((H_sel >= H_high) & (G_sel <= Y_low_thr))
ok_outage2 = ~((E_sel >= E_high) & (G_sel <= Y_low_thr))

outage_ok = ok_night & ok_outage & ok_outage2

mask_outage = pd.Series(True, index=df0.index, dtype=bool)
mask_outage.loc[sel_idx] = outage_ok.values
mask_keep = apply_rule(mask_outage, "outage_like_days")

def fit_iqr_thresholds_within_bins(y, x, bins=10, k=3.5, use_log_y=True, min_bin_size=25):
    """
    Fit IQR thresholds by x-bins using ONLY calibration subset.
    Returns a list of dicts: [{"left":..., "right":..., "low":..., "high":...}, ...]
    """
    tmp = pd.DataFrame({
        "y": pd.to_numeric(y, errors="coerce"),
        "x": pd.to_numeric(x, errors="coerce")
    }).dropna()

    if len(tmp) == 0:
        return []

    try:
        tmp["bin"] = pd.qcut(tmp["x"], q=bins, duplicates="drop")
    except Exception:
        tmp["bin"] = "all"

    thresholds = []
    for bin_name, grp in tmp.groupby("bin", observed=False):
        if len(grp) < min_bin_size:
            continue

        yy = grp["y"].to_numpy(dtype=float)
        if use_log_y:
            yy = np.log1p(np.clip(yy, 0, None))

        q1, q3 = np.nanpercentile(yy, [25, 75])
        iqr = q3 - q1
        low = q1 - k * iqr
        high = q3 + k * iqr

        if isinstance(bin_name, str) and bin_name == "all":
            left, right = -np.inf, np.inf
        else:
            left = float(bin_name.left)
            right = float(bin_name.right)

        thresholds.append({
            "left": left,
            "right": right,
            "low": float(low),
            "high": float(high)
        })

    return thresholds

def apply_iqr_thresholds(y, x, thresholds, use_log_y=True):
    """
    Apply already-fit thresholds to arbitrary subset.
    Safe index handling; no fragile get_indexer logic.
    """
    tmp = pd.DataFrame({
        "y": pd.to_numeric(y, errors="coerce"),
        "x": pd.to_numeric(x, errors="coerce")
    }, index=y.index)

    ok = pd.Series(True, index=tmp.index, dtype="boolean")

    if len(thresholds) == 0:
        return ok.fillna(True).astype(bool)

    vals = tmp["y"].to_numpy(dtype=float)
    if use_log_y:
        vals = np.log1p(np.clip(vals, 0, None))
    tmp["_yy"] = vals

    covered = pd.Series(False, index=tmp.index, dtype="boolean")

    for th in thresholds:
        left, right = th["left"], th["right"]
        low, high = th["low"], th["high"]

        m = (tmp["x"] > left) & (tmp["x"] <= right)
        if m.any():
            ok.loc[m] = (tmp.loc[m, "_yy"] >= low) & (tmp.loc[m, "_yy"] <= high)
            covered.loc[m] = True

    ok.loc[~covered.fillna(False)] = True
    return ok.fillna(True).astype(bool)

alive_train_idx = [i for i in proxy_train_idx if mask_keep.loc[i]]
H_train_alive = df0.loc[alive_train_idx, "solarenergy"]
G_train_alive = df0.loc[alive_train_idx, "Generated"]

iqr_thresholds = fit_iqr_thresholds_within_bins(
    y=G_train_alive,
    x=H_train_alive,
    bins=10,
    k=3.5,
    use_log_y=True,
    min_bin_size=25
)

sel_idx = mask_keep[mask_keep].index
H_sel = df0.loc[sel_idx, "solarenergy"]
G_sel = df0.loc[sel_idx, "Generated"]

iqr_ok_local = apply_iqr_thresholds(
    y=G_sel,
    x=H_sel,
    thresholds=iqr_thresholds,
    use_log_y=True
)

iqr_ok = pd.Series(True, index=df0.index, dtype=bool)
iqr_ok.loc[sel_idx] = iqr_ok_local.astype(bool).values
mask_keep = apply_rule(iqr_ok, "iqr_outliers_generated_solar")

df_clean = df0.loc[mask_keep].copy().reset_index(drop=True)

if "E_phys_tmp_kWh" in df_clean.columns:
    df_clean.drop(columns=["E_phys_tmp_kWh"], inplace=True)

df_clean.to_csv(OUT_PATH, index=False)

print(f"🧹 Removed {int((~mask_keep).sum())} rows in total.")
print("Details (rule → rows removed; rules are applied sequentially):")
for k, v in report.items():
    print(f"  - {k}: {v}")

print(f"\n💾 Saved: {OUT_PATH} | Rows: {len(df_clean)}")
print("\n[Train-calibrated thresholds]")
print(f"H_high (q={H_HIGH_Q}) = {H_high:.3f} kWh/m²/day")
print(f"E_high (q={E_HIGH_Q}) = {E_high:.3f} kWh/day")
print(f"Y_low_thr = {Y_low_thr:.3f} kWh/day")
print(f"Night rule: H < {H_MIN_NIGHT} and Gen > {Y_MIN_NIGHT}")
print(f"Over-prod factor = {OVER_FACTOR:.3f} | MIN_EPHYS = {MIN_EPHYS}")


In [ ]:
# Feature preparation and tensor construction

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch

df = pd.read_csv("Dataset_PINN_Input_filtered.csv", parse_dates=["datetime"]).sort_values("datetime").reset_index(drop=True)

num_cols = [
    "Generated","solarenergy","tempmax","temp","humidity","cloudcover",
    "precipcover","windspeed","Day_Length","DOY_sin","DOY_cos","humidity_wind"
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

target = "Generated"
assert target in df.columns, "Target column 'Generated' was not found in Dataset_PINN_Input_filtered.csv"

if ("DOY_sin" not in df.columns) or ("DOY_cos" not in df.columns):
    doy = df["datetime"].dt.dayofyear
    df["DOY_sin"] = np.sin(2*np.pi*doy/365.0)
    df["DOY_cos"] = np.cos(2*np.pi*doy/365.0)

df["month"] = df["datetime"].dt.month
df["Month_sin"] = np.sin(2*np.pi*df["month"]/12.0)
df["Month_cos"] = np.cos(2*np.pi*df["month"]/12.0)

if ("humidity_wind" not in df.columns) and {"humidity","windspeed"}.issubset(df.columns):
    df["humidity_wind"] = df["humidity"] * df["windspeed"]

# Keep weather conditions as categorical features and one-hot encode them later.
cond_ohe_cols = []
if "conditions" in df.columns:
    cond = df["conditions"].fillna("Unknown").astype(str).str.strip()

    top_conditions = cond.value_counts().head(6).index.tolist()
    cond_small = cond.where(cond.isin(top_conditions), other="Other")

    cond_dummies = pd.get_dummies(cond_small, prefix="cond", dtype=np.float32)
    df = pd.concat([df, cond_dummies], axis=1)
    cond_ohe_cols = cond_dummies.columns.tolist()

df["Generated_shift1"] = df["Generated"].shift(1)

ewm_cols = ["tempmax","humidity","solarenergy","Generated"]
for col in ewm_cols:
    df[f"{col}_ewm3"] = df[col].shift(1).ewm(span=3, adjust=False).mean()
    df[f"{col}_ewm7"] = df[col].shift(1).ewm(span=7, adjust=False).mean()

A_total_m2 = 65.0
eta_ref    = 0.15
NOCT       = 45.0
gamma_p    = -0.004
c_wind     = 0.7
PR_base    = 0.82

df["G_avg"] = np.clip(df["solarenergy"] / 24.0, 0.0, 1.2)

Tamb = df["tempmax"].astype(float)
wind = df["windspeed"].fillna(0).astype(float)

df["T_cell_est"] = Tamb + (NOCT - 20.0) * df["G_avg"] - c_wind * wind
df["f_T_est"]    = np.clip(1.0 + gamma_p * (df["T_cell_est"] - 25.0), 0.70, 1.15)
df["E_dc_est"]   = np.clip(df["solarenergy"] * A_total_m2 * eta_ref * df["f_T_est"], 0.0, None)
df["PR_est_base"] = PR_base

drop_needed = [
    target, "solarenergy", "tempmax", "Generated_shift1",
    "humidity", "cloudcover", "windspeed", "precipcover", "Day_Length",
    "DOY_sin", "DOY_cos", "Month_sin", "Month_cos",
    "G_avg", "T_cell_est", "f_T_est", "E_dc_est"
]
df = df.dropna(subset=drop_needed).reset_index(drop=True)

# This pipeline uses target-day weather covariates.
base_features = [
    "tempmax","temp","humidity","cloudcover","solarenergy","precipcover",
    "Day_Length","windspeed",
    "DOY_sin","DOY_cos","Month_sin","Month_cos",
    "humidity_wind","Generated_shift1",

    "tempmax_ewm3","tempmax_ewm7",
    "humidity_ewm3","humidity_ewm7",
    "solarenergy_ewm3","solarenergy_ewm7",
    "Generated_ewm3","Generated_ewm7",

    "G_avg","T_cell_est","f_T_est","E_dc_est","PR_est_base",
] + cond_ohe_cols

missing = [c for c in base_features if c not in df.columns]
if missing:
    print("⚠️ Dropping missing columns:", missing)
    base_features = [c for c in base_features if c in df.columns]

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch

sequence_length_past = 14
include_target_in_seq = True
S = sequence_length_past + (1 if include_target_in_seq else 0)

X_seq, y_seq, dates = [], [], []
for i in range(sequence_length_past, len(df)):
    start = i - sequence_length_past
    end = i + 1 if include_target_in_seq else i

    x_window = df.iloc[start:end][base_features].values
    if include_target_in_seq and x_window.shape[0] != S:
        continue

    X_seq.append(x_window)
    y_seq.append(df.iloc[i][target])
    dates.append(df.iloc[i]["datetime"])

X_seq = np.array(X_seq, dtype=np.float32)
y_seq = np.array(y_seq, dtype=np.float32).reshape(-1, 1)
dates = pd.to_datetime(dates)

print("Forma e inputeve:", X_seq.shape, "| F =", len(base_features))
print("Forma e targeteve:", y_seq.shape)
print("Datat:", dates.min(), "→", dates.max())
print("Task mode: daily prediction with target-day weather covariates")

PRED_START = pd.Timestamp("2025-08-30")

mask_measured = dates < PRED_START
mask_forecasted = dates >= PRED_START

X_seq_measured = X_seq[mask_measured]
y_seq_measured = y_seq[mask_measured]
dates_measured = dates[mask_measured]

X_seq_forecasted = X_seq[mask_forecasted]
y_seq_forecasted = y_seq[mask_forecasted]
dates_forecasted = dates[mask_forecasted]

print("\n=== Regime split ===")
print("Measured:", X_seq_measured.shape, y_seq_measured.shape,
      "|", dates_measured.min() if len(dates_measured) else None,
      "->", dates_measured.max() if len(dates_measured) else None)
print("Forecasted:", X_seq_forecasted.shape, y_seq_forecasted.shape,
      "|", dates_forecasted.min() if len(dates_forecasted) else None,
      "->", dates_forecasted.max() if len(dates_forecasted) else None)

num_samples_measured, S_eff, num_features = X_seq_measured.shape

train_end = int(num_samples_measured * 0.75)
val_end   = int(num_samples_measured * 0.90)

X_train_raw = X_seq_measured[:train_end]
y_train_raw = y_seq_measured[:train_end]
dates_train = dates_measured[:train_end]

X_val_raw = X_seq_measured[train_end:val_end]
y_val_raw = y_seq_measured[train_end:val_end]
dates_val = dates_measured[train_end:val_end]

X_main_test_raw = X_seq_measured[val_end:]
y_main_test_raw = y_seq_measured[val_end:]
dates_main_test = dates_measured[val_end:]

X_pred_test_raw = X_seq_forecasted
y_pred_test_true_raw = y_seq_forecasted
dates_pred_test = dates_forecasted

print("\n=== Final split ===")
print("Train      :", X_train_raw.shape, y_train_raw.shape, "|",
      dates_train.min() if len(dates_train) else None, "->",
      dates_train.max() if len(dates_train) else None)
print("Val        :", X_val_raw.shape, y_val_raw.shape, "|",
      dates_val.min() if len(dates_val) else None, "->",
      dates_val.max() if len(dates_val) else None)
print("Main test  :", X_main_test_raw.shape, y_main_test_raw.shape, "|",
      dates_main_test.min() if len(dates_main_test) else None, "->",
      dates_main_test.max() if len(dates_main_test) else None)
print("Pred test  :", X_pred_test_raw.shape, y_pred_test_true_raw.shape, "|",
      dates_pred_test.min() if len(dates_pred_test) else None, "->",
      dates_pred_test.max() if len(dates_pred_test) else None)

scaler_X = StandardScaler()
X_train_2d = X_train_raw.reshape(-1, X_train_raw.shape[2])
scaler_X.fit(X_train_2d)

X_train = scaler_X.transform(X_train_2d).reshape(X_train_raw.shape)
X_val   = scaler_X.transform(X_val_raw.reshape(-1, X_val_raw.shape[2])).reshape(X_val_raw.shape)

X_main_test = scaler_X.transform(
    X_main_test_raw.reshape(-1, X_main_test_raw.shape[2])
).reshape(X_main_test_raw.shape) if len(X_main_test_raw) else np.empty((0, S_eff, num_features), dtype=np.float32)

X_pred_test = scaler_X.transform(
    X_pred_test_raw.reshape(-1, X_pred_test_raw.shape[2])
).reshape(X_pred_test_raw.shape) if len(X_pred_test_raw) else np.empty((0, S_eff, num_features), dtype=np.float32)

y_train_log = np.log1p(y_train_raw)
y_val_log   = np.log1p(y_val_raw)

y_main_test_log = np.log1p(y_main_test_raw) if len(y_main_test_raw) else np.empty((0, 1), dtype=np.float32)
y_pred_test_true_log = np.log1p(y_pred_test_true_raw) if len(y_pred_test_true_raw) else np.empty((0, 1), dtype=np.float32)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train_log)
y_val   = scaler_y.transform(y_val_log)

y_main_test = scaler_y.transform(y_main_test_log) if len(y_main_test_log) else np.empty((0, 1), dtype=np.float32)
y_pred_test_true = scaler_y.transform(y_pred_test_true_log) if len(y_pred_test_true_log) else np.empty((0, 1), dtype=np.float32)

assert "solarenergy" in base_features and "tempmax" in base_features, "solarenergy and tempmax are required!"
IDX_GHI  = base_features.index("solarenergy")
IDX_TAMB = base_features.index("tempmax")
IDX_WIND = base_features.index("windspeed") if "windspeed" in base_features else None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

X_main_test_tensor = torch.tensor(X_main_test, dtype=torch.float32)
y_main_test_tensor = torch.tensor(y_main_test, dtype=torch.float32)

X_pred_test_tensor = torch.tensor(X_pred_test, dtype=torch.float32)
y_pred_test_true_tensor = torch.tensor(y_pred_test_true, dtype=torch.float32)

month_train_tensor = torch.tensor(dates_train.month.astype(np.int64).to_numpy(), dtype=torch.long)
month_val_tensor = torch.tensor(dates_val.month.astype(np.int64).to_numpy(), dtype=torch.long)
month_main_test_tensor = torch.tensor(dates_main_test.month.astype(np.int64).to_numpy(), dtype=torch.long)
month_pred_test_tensor = torch.tensor(dates_pred_test.month.astype(np.int64).to_numpy(), dtype=torch.long)

feat_means  = torch.tensor(scaler_X.mean_.astype(np.float32),  device=device)
feat_scales = torch.tensor(scaler_X.scale_.astype(np.float32), device=device)

print("\nMonth tensors:",
      month_train_tensor.shape,
      month_val_tensor.shape,
      month_main_test_tensor.shape,
      month_pred_test_tensor.shape)

print("IDX_GHI =", IDX_GHI, "IDX_TAMB =", IDX_TAMB, "IDX_WIND =", IDX_WIND, "| S =", S_eff)
print("Train shapes     :", X_train_tensor.shape, y_train_tensor.shape)
print("Val shapes       :", X_val_tensor.shape, y_val_tensor.shape)
print("Main test shapes :", X_main_test_tensor.shape, y_main_test_tensor.shape)
print("Pred test shapes :", X_pred_test_tensor.shape, y_pred_test_true_tensor.shape)


In [ ]:
# Install optional training utilities

!rm -f lookahead.py
!pip install torch-optimizer --quiet
!pip install torch-ema --quiet


In [ ]:
# Evaluation utilities for static and rolling test splits

import torch, numpy as np

BATCH_EVAL = 4096
ROLL_WIN_DEFAULT = 15
ROLL_STEP_DEFAULT = 1

@torch.no_grad()
def scaled_to_real_torch(y_scaled: torch.Tensor, model, to_cpu: bool = False) -> torch.Tensor:
    dev = model.y_log_std.device
    y_log = y_scaled.to(dev) * model.y_log_std + model.y_log_mean
    y_real = torch.expm1(y_log)
    return y_real.cpu() if to_cpu else y_real

@torch.no_grad()
def compute_metrics(y_true: torch.Tensor, y_hat: torch.Tensor):
    y_true = y_true.float()
    y_hat  = y_hat.float()
    err = y_true - y_hat

    mae  = torch.mean(torch.abs(err)).item()
    rmse = torch.sqrt(torch.mean(err**2)).item()

    ybar = torch.mean(y_true).item()
    ss_res = torch.sum((y_true - y_hat)**2).item()
    ss_tot = torch.sum((y_true - ybar)**2).item() + 1e-12
    r2 = 1.0 - ss_res/ss_tot

    mask = y_true > 1.0
    mape = float((torch.abs(y_true[mask]-y_hat[mask])/(y_true[mask]+1e-9)).mean()*100.0) if mask.sum()>0 else float("nan")
    smape = float((2.0*torch.abs(y_true - y_hat)/(torch.abs(y_true)+torch.abs(y_hat)+1e-9)).mean()*100.0)
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE>1": mape, "SMAPE": smape}

@torch.no_grad()
def model_predict_real(model, X, month_idx=None, batch_size=4096, use_ema=True, ema=None):
    """
    EMA-first predictor.
    Assumes model(...) returns dict with key 'y_real'.
    """
    device = next(model.parameters()).device

    backup = None
    if use_ema and (ema is not None):
        backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
        ema.load_shadow(model)

    model.eval()
    ys = []
    N = int(X.shape[0])
    for i in range(0, N, batch_size):
        xb = X[i:i+batch_size].to(device)
        if month_idx is None:
            out = model(xb)
        else:
            mb = month_idx[i:i+batch_size].to(device).view(-1).long()
            out = model(xb, month_idx=mb)

        if "y_real" not in out:
            raise KeyError("Model output must contain key 'y_real'.")
        ys.append(out["y_real"].squeeze(-1).detach().cpu())

    if backup is not None:
        model.load_state_dict(backup, strict=True)

    return torch.cat(ys, dim=0)

@torch.no_grad()
def predict_y_phys_from_hybrid(model, X, month, batch_size=4096, use_ema=True, ema=None):
    """
    PHYS baseline extracted from HYBRID output: 'E_ac' or 'y_real_phys'
    """
    device = next(model.parameters()).device

    backup = None
    if use_ema and (ema is not None):
        backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
        ema.load_shadow(model)

    model.eval()
    ys = []
    N = int(X.shape[0])
    for i in range(0, N, batch_size):
        xb = X[i:i+batch_size].to(device)
        mb = month[i:i+batch_size].to(device).view(-1).long()
        out = model(xb, month_idx=mb)

        if "E_ac" in out:
            y_phys = out["E_ac"].squeeze(-1)
        elif "y_real_phys" in out:
            y_phys = out["y_real_phys"].squeeze(-1)
        else:
            raise KeyError("Hybrid output missing 'E_ac'/'y_real_phys'.")
        ys.append(y_phys.detach().cpu())

    if backup is not None:
        model.load_state_dict(backup, strict=True)

    return torch.cat(ys, dim=0)

@torch.no_grad()
def eval_main_pred_score(
    model,
    X_main, y_main, month_main,
    X_pred, y_pred, month_pred,
    lambda_score=1.0,
    batch_size=4096,
    name="MODEL",
    use_ema=True,
    ema=None,
    also_phys=False
):
    """
    Static evaluation on:
      - MAIN_TEST
      - PRED_TEST
    Returns:
      metrics_main, metrics_pred, dRMSE, score
    """
    y_true_main = scaled_to_real_torch(y_main, model, to_cpu=True).squeeze(-1).detach()
    y_pred_main = model_predict_real(model, X_main, month_idx=month_main, batch_size=batch_size, use_ema=use_ema, ema=ema)

    y_true_pred = scaled_to_real_torch(y_pred, model, to_cpu=True).squeeze(-1).detach()
    y_pred_pred = model_predict_real(model, X_pred, month_idx=month_pred, batch_size=batch_size, use_ema=use_ema, ema=ema)

    m_main = compute_metrics(y_true_main, y_pred_main)
    m_pred = compute_metrics(y_true_pred, y_pred_pred)

    d_rmse = m_pred["RMSE"] - m_main["RMSE"]
    score  = m_main["RMSE"] + float(lambda_score) * m_pred["RMSE"]

    print(f"\n=== {name} | EMA={bool(use_ema and (ema is not None))} ===")
    print(f"MAIN  (N={len(X_main)}) | RMSE={m_main['RMSE']:.4f} | MAE={m_main['MAE']:.4f} | R2={m_main['R2']:.4f} | MAPE>1={m_main['MAPE>1']:.3f}%")
    print(f"PRED  (N={len(X_pred)}) | RMSE={m_pred['RMSE']:.4f} | MAE={m_pred['MAE']:.4f} | R2={m_pred['R2']:.4f} | MAPE>1={m_pred['MAPE>1']:.3f}%")
    print(f"ΔRMSE (pred-main) = {d_rmse:+.4f}")
    print(f"SCORE (RMSE_main + {lambda_score}*RMSE_pred) = {score:.4f}")

    out = {
        "metrics_main": m_main,
        "metrics_pred": m_pred,
        "dRMSE": d_rmse,
        "score": score,
        "y_true_main": y_true_main,
        "y_pred_main": y_pred_main,
        "y_true_pred": y_true_pred,
        "y_pred_pred": y_pred_pred,
    }

    if also_phys:
        y_phys_main = predict_y_phys_from_hybrid(model, X_main, month_main, batch_size=batch_size, use_ema=use_ema, ema=ema)
        y_phys_pred = predict_y_phys_from_hybrid(model, X_pred, month_pred, batch_size=batch_size, use_ema=use_ema, ema=ema)

        mp_main = compute_metrics(y_true_main, y_phys_main)
        mp_pred = compute_metrics(y_true_pred, y_phys_pred)

        dp_rmse = mp_pred["RMSE"] - mp_main["RMSE"]
        sp = mp_main["RMSE"] + float(lambda_score) * mp_pred["RMSE"]

        print(f"\n--- PHYS baseline (from Hybrid) ---")
        print(f"MAIN | RMSE={mp_main['RMSE']:.4f} | MAE={mp_main['MAE']:.4f} | R2={mp_main['R2']:.4f}")
        print(f"PRED | RMSE={mp_pred['RMSE']:.4f} | MAE={mp_pred['MAE']:.4f} | R2={mp_pred['R2']:.4f}")
        print(f"ΔRMSE_phys = {dp_rmse:+.4f} | SCORE_phys={sp:.4f}")

        out["phys_main"] = mp_main
        out["phys_pred"] = mp_pred
        out["phys_dRMSE"] = dp_rmse
        out["phys_score"] = sp
        out["y_phys_main"] = y_phys_main
        out["y_phys_pred"] = y_phys_pred

    return out

@torch.no_grad()
def rolling_eval(model, X_seg, y_seg, month_seg, win=15, step=1, batch_size=4096, use_ema=True, ema=None, label="SEGMENT"):
    """
    Rolling evaluation on a single chronologically ordered segment.
    Example:
      - MAIN_TEST only
      - PRED_TEST only
    """
    N = int(X_seg.shape[0])
    assert N >= win, f"{label} too short N={N} for win={win}"

    y_true = scaled_to_real_torch(y_seg, model, to_cpu=True).squeeze(-1).detach()
    y_pred = model_predict_real(model, X_seg, month_idx=month_seg, batch_size=batch_size, use_ema=use_ema, ema=ema)

    rmses, maes, r2s = [], [], []
    starts = list(range(0, N-win+1, step))

    for s in starts:
        e = s + win
        m = compute_metrics(y_true[s:e], y_pred[s:e])
        rmses.append(m["RMSE"])
        maes.append(m["MAE"])
        r2s.append(m["R2"])

    rmses = np.array(rmses, dtype=np.float64)
    maes  = np.array(maes,  dtype=np.float64)
    r2s   = np.array(r2s,   dtype=np.float64)

    summary = {
        "count": int(len(rmses)),
        "rmse_mean": float(np.mean(rmses)),
        "rmse_median": float(np.median(rmses)),
        "rmse_p90": float(np.quantile(rmses, 0.90)),
        "rmse_p10": float(np.quantile(rmses, 0.10)),
        "mae_mean": float(np.mean(maes)),
        "r2_mean": float(np.mean(r2s)),
    }

    print(f"\n--- Rolling {label} summary ---")
    print(f"windows={summary['count']} | RMSE mean={summary['rmse_mean']:.4f} | median={summary['rmse_median']:.4f} | p90={summary['rmse_p90']:.4f}")
    print(f"MAE mean={summary['mae_mean']:.4f} | R2 mean={summary['r2_mean']:.4f}")

    return {
        "starts": np.array(starts),
        "rmse": rmses,
        "mae": maes,
        "r2": r2s,
        "summary": summary
    }


In [ ]:
# Physics-augmented CNN-BiLSTM-Attention model

import math
from typing import Optional, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNBiLSTMAttnEncoder(nn.Module):
    def __init__(
        self,
        in_dim: int,
        cnn_channels: int = 64,
        kernel_size: int = 3,
        lstm_hidden: int = 64,
        lstm_layers: int = 1,
        dropout: float = 0.10,
    ):
        super().__init__()
        pad = kernel_size // 2

        self.cnn = nn.Sequential(
            nn.Conv1d(in_dim, cnn_channels, kernel_size=kernel_size, padding=pad),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Conv1d(cnn_channels, cnn_channels, kernel_size=kernel_size, padding=pad),
            nn.SiLU(),
        )

        self.lstm = nn.LSTM(
            input_size=cnn_channels,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=(dropout if lstm_layers > 1 else 0.0),
        )

        attn_dim = 2 * lstm_hidden
        self.attn_proj = nn.Linear(attn_dim, attn_dim)
        self.attn_score = nn.Linear(attn_dim, 1)

        self.out_dim = attn_dim

    def forward(self, X: torch.Tensor):
        """
        X: [N,S,F]
        returns:
          ctx: [N, 2*lstm_hidden]
          w:   [N,S,1]
        """
        x = X.transpose(1, 2)           # [N,F,S]
        x = self.cnn(x)                 # [N,C,S]
        x = x.transpose(1, 2)           # [N,S,C]
        h, _ = self.lstm(x)             # [N,S,2H]

        a = torch.tanh(self.attn_proj(h))
        s = self.attn_score(a)          # [N,S,1]
        w = torch.softmax(s, dim=1)     # [N,S,1]
        ctx = torch.sum(w * h, dim=1)   # [N,2H]
        return ctx, w

class PhysicsPVDailyMeaningfulMonthlyPR(nn.Module):
    def __init__(
        self,
        idx_h: int,
        idx_tamb: int,
        x_means: torch.Tensor,
        x_scales: torch.Tensor,
        idx_wind: Optional[int] = None,
        idx_solarrad: Optional[int] = None,
        A_total_m2_init: float = 65.0,
        eta_ref_init: float = 0.15,
        PR_base_init: float = 0.82,
        eta_inv_init: float = 0.97,
        NOCT_init: float = 45.0,
        gamma_p_init: float = -0.0040,
        c_wind_init: float = 0.7,
        pr_month_init: Optional[torch.Tensor] = None,  # (12,)
        bias_init: float = 0.0,
        trainable: bool = True,
    ):
        super().__init__()
        self.idx_h = int(idx_h)
        self.idx_tamb = int(idx_tamb)
        self.idx_wind = int(idx_wind) if idx_wind is not None else None
        self.idx_solarrad = int(idx_solarrad) if idx_solarrad is not None else None

        self.register_buffer("x_means",  x_means.clone().detach())
        self.register_buffer("x_scales", x_scales.clone().detach())

        def P(v: float):
            return nn.Parameter(torch.tensor(float(v), dtype=torch.float32), requires_grad=trainable)

        self.A_total = P(A_total_m2_init)
        self.eta_ref = P(eta_ref_init)
        self.PR_base = P(PR_base_init)
        self.eta_inv = P(eta_inv_init)
        self.NOCT    = P(NOCT_init)
        self.gamma_p = P(gamma_p_init)
        self.c_wind  = P(c_wind_init)

        if pr_month_init is None:
            init = torch.ones(12, dtype=torch.float32)
        else:
            init = pr_month_init.detach().float().view(12).clone()
        self.pr_month = nn.Parameter(init, requires_grad=trainable)

        self.physics_bias = P(bias_init)

    def _denorm_2d(self, X2: torch.Tensor, idx: int) -> torch.Tensor:
        return X2[:, idx] * self.x_scales[idx] + self.x_means[idx]

    def _compute_from_Xlast2d(
        self,
        X_last2d: torch.Tensor,
        month_idx: Optional[torch.Tensor] = None,
        pr_dyn: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        """
        Computes physics using a 2D "last-step" input: X_last2d [N,F]
        Returns tensors shaped [N,1].
        """
        H = torch.clamp(self._denorm_2d(X_last2d, self.idx_h), min=0.0)
        T_amb = self._denorm_2d(X_last2d, self.idx_tamb)

        if self.idx_wind is not None:
            wind = torch.clamp(self._denorm_2d(X_last2d, self.idx_wind), min=0.0)
        else:
            wind = torch.zeros_like(H)

        if self.idx_solarrad is not None:
            SR = torch.clamp(self._denorm_2d(X_last2d, self.idx_solarrad), min=0.0)
            G_avg = torch.clamp(SR / 1000.0, 0.0, 1.2)
        else:
            G_avg = torch.clamp(H / 24.0, 0.0, 1.2)

        A    = torch.clamp(self.A_total,  10.0, 500.0)
        eta  = torch.clamp(self.eta_ref,  0.05, 0.25)
        prb0 = torch.clamp(self.PR_base,  0.60, 0.98)
        einv = torch.clamp(self.eta_inv,  0.85, 0.995)
        noct = torch.clamp(self.NOCT,    30.0, 60.0)
        gam  = torch.clamp(self.gamma_p, -0.01, -0.0005)
        cw   = torch.clamp(self.c_wind,  0.0, 5.0)

        if month_idx is not None:
            mi = month_idx.view(-1).long().clamp(1, 12) - 1
            pr_m = torch.clamp(self.pr_month[mi], 0.70, 1.30)
        else:
            pr_m = torch.ones_like(H)

        pr_h = torch.ones_like(H)  # disabled
        prb = torch.clamp(prb0 * pr_m * pr_h, 0.50, 1.20)

        T_cell = T_amb + (noct - 20.0) * G_avg - cw * wind
        f_T = torch.clamp(1.0 + gam * (T_cell - 25.0), 0.70, 1.15)

        E_dc = torch.clamp(H * A * eta * f_T, min=0.0)

        if pr_dyn is not None:
            pr_dyn = pr_dyn.view(-1) if pr_dyn.dim() > 1 else pr_dyn
            pr_dyn = torch.clamp(pr_dyn, 0.97, 1.03)  # narrower than 0.95..1.05
            PR_total = torch.clamp(prb * pr_dyn, 0.50, 1.20)
        else:
            PR_total = prb

        bias = torch.clamp(self.physics_bias, -0.2, 0.2)
        E_ac = torch.clamp(E_dc * einv * PR_total, min=0.0) * (1.0 + bias)

        return {
            "E_dc": E_dc.unsqueeze(-1),
            "E_ac": E_ac.unsqueeze(-1),
            "T_cell": T_cell.unsqueeze(-1),
            "f_T": f_T.unsqueeze(-1),
            "PR_total": PR_total.unsqueeze(-1),
            "PR_month": pr_m.unsqueeze(-1),
            "PR_H": pr_h.unsqueeze(-1),
            "physics_bias": bias.unsqueeze(0),
            "A_total_m2": A.detach(),
            "eta_ref": eta.detach(),
            "eta_inv": einv.detach(),
            "PR_base": prb0.detach(),
            "NOCT": noct.detach(),
            "gamma_p": gam.detach(),
            "c_wind": cw.detach(),
        }

    def forward(
        self,
        X: torch.Tensor,
        pr_dyn: Optional[torch.Tensor] = None,
        month_idx: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        X_last = X[:, -1, :]
        return self._compute_from_Xlast2d(X_last, month_idx=month_idx, pr_dyn=pr_dyn)

    @torch.no_grad()
    def sequence_feats(
        self,
        X: torch.Tensor,
        month_idx: Optional[torch.Tensor],
    ) -> Dict[str, torch.Tensor]:
        """
        Variant B: compute physics-derived features for EACH timestep.
        We reuse the target-day month_idx for all steps (window is short; stable, simple).
        Returns:
          feats_seq: [N,S,4] where columns are:
            E_dc_feat, fT_feat, PR_feat, T_cell_feat
        """
        N, S, F = X.shape
        X2 = X.reshape(N * S, F)

        if month_idx is not None:
            m2 = month_idx.view(-1).repeat_interleave(S)   # [N*S]
        else:
            m2 = None

        phys2 = self._compute_from_Xlast2d(X2, month_idx=m2, pr_dyn=None)

        E_dc_feat = phys2["E_dc"] / 60.0          # ~O(1)
        fT_feat   = phys2["f_T"] - 1.0            # centered
        PR_feat   = phys2["PR_total"] - 0.85      # centered
        Tcell_feat = (phys2["T_cell"] - 25.0) / 20.0

        feats2 = torch.cat([E_dc_feat, fT_feat, PR_feat, Tcell_feat], dim=-1)  # [N*S,4]
        feats_seq = feats2.reshape(N, S, 4)
        return {"feats_seq": feats_seq}

class HybridPINNMeaningful(nn.Module):
    def __init__(
        self,
        num_features: int,
        idx_h: int,
        idx_tamb: int,
        x_means: torch.Tensor,
        x_scales: torch.Tensor,
        y_log_mean: float,
        y_log_std: float,
        idx_wind: Optional[int] = None,
        idx_solarrad: Optional[int] = None,
        cnn_channels: int = 64,
        kernel_size: int = 3,
        lstm_hidden: int = 64,
        lstm_layers: int = 1,
        seq_dropout: float = 0.10,
        pr_dyn_enabled: bool = False,
        residual_max: float = 0.50,
    ):
        super().__init__()
        self.register_buffer("y_log_mean", torch.tensor([float(y_log_mean)], dtype=torch.float32))
        self.register_buffer("y_log_std",  torch.tensor([float(y_log_std)],  dtype=torch.float32))

        pr_month_init = torch.tensor(
            [1.06, 0.98, 0.82, 0.76, 0.70, 0.73, 0.73, 0.81, 0.90, 1.03, 1.11, 1.08],
            dtype=torch.float32, device=self.y_log_mean.device
        )

        self.physics = PhysicsPVDailyMeaningfulMonthlyPR(
            idx_h=idx_h,
            idx_tamb=idx_tamb,
            x_means=x_means,
            x_scales=x_scales,
            idx_wind=idx_wind,
            idx_solarrad=idx_solarrad,
            pr_month_init=pr_month_init,
            trainable=True,
        )

        self.aug_dim = 4
        self.seq_encoder = CNNBiLSTMAttnEncoder(
            in_dim=num_features + self.aug_dim,
            cnn_channels=cnn_channels,
            kernel_size=kernel_size,
            lstm_hidden=lstm_hidden,
            lstm_layers=lstm_layers,
            dropout=seq_dropout,
        )
        ctx_dim = self.seq_encoder.out_dim

        self.pr_dyn_enabled = bool(pr_dyn_enabled)
        if self.pr_dyn_enabled:
            self.pr_head = nn.Sequential(
                nn.Linear(2 * num_features, 128),
                nn.SiLU(),
                nn.Linear(128, 1),
            )
        else:
            self.pr_head = None

        resid_in = 2 * num_features + ctx_dim + 2 + 3
        self.residual = nn.Sequential(
            nn.Linear(resid_in, 512),
            nn.LayerNorm(512),
            nn.SiLU(),
            nn.Dropout(0.15),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Dropout(0.10),
            nn.Linear(256, 64),
            nn.SiLU(),
            nn.Linear(64, 1),
        )

        self.residual_max = float(residual_max)
        self.calib_a = nn.Parameter(torch.tensor(0.0))
        self.calib_b = nn.Parameter(torch.tensor(1.0))

    def _scaled_to_real(self, y_scaled: torch.Tensor) -> torch.Tensor:
        y_log = y_scaled * self.y_log_std + self.y_log_mean
        return torch.expm1(y_log)

    def _real_to_scaled(self, y_real: torch.Tensor) -> torch.Tensor:
        y_log = torch.log1p(y_real.clamp(min=0.0))
        return (y_log - self.y_log_mean) / (self.y_log_std + 1e-12)

    def forward(self, X: torch.Tensor, month_idx: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        x_last = X[:, -1, :]
        x_mean = X.mean(dim=1)

        pr_dyn = None
        if self.pr_dyn_enabled:
            pr_raw = self.pr_head(torch.cat([x_last, x_mean], dim=-1))
            pr_dyn = 0.95 + 0.10 * torch.sigmoid(pr_raw)
            pr_dyn = pr_dyn.squeeze(-1)

        phys = self.physics(X, pr_dyn=pr_dyn, month_idx=month_idx)
        E_ac = phys["E_ac"]                       # [N,1]
        y_scaled_phys = self._real_to_scaled(E_ac)

        phys_seq = self.physics.sequence_feats(X, month_idx=month_idx)["feats_seq"]  # [N,S,4]
        X_aug = torch.cat([X, phys_seq], dim=-1)                                   # [N,S,F+4]

        ctx, attn_w = self.seq_encoder(X_aug)                                       # ctx [N,ctx_dim]

        T_cell = phys["T_cell"]                    # [N,1]
        T_cell_feat = (T_cell - 25.0) / 20.0

        E_dc_feat = phys["E_dc"] / 60.0
        fT_feat   = phys["f_T"] - 1.0
        PR_feat   = phys["PR_total"] - 0.85
        phys_feats = torch.cat([E_dc_feat, fT_feat, PR_feat], dim=-1)  # [N,3]

        feats = torch.cat([x_last, x_mean, ctx, y_scaled_phys, T_cell_feat, phys_feats], dim=-1)
        r = self.residual(feats)
        r = self.residual_max * torch.tanh(r)     # clamp correction

        y_scaled_raw = y_scaled_phys + r
        y_scaled = self.calib_a + self.calib_b * y_scaled_raw
        y_real = self._scaled_to_real(y_scaled).clamp(min=0.0)

        out = {
            "y_real": y_real,
            "y_scaled": y_scaled,
            "y_real_phys": E_ac,
            "E_dc": phys["E_dc"],
            "E_ac": phys["E_ac"],
            "T_cell": phys["T_cell"],
            "f_T": phys["f_T"],
            "PR_total": phys["PR_total"],
            "PR_month": phys["PR_month"],
            "PR_H": phys["PR_H"],
            "attn": attn_w,   # [N,S,1]
            "extra": {
                "A_total_m2": phys["A_total_m2"],
                "eta_ref": phys["eta_ref"],
                "eta_inv": phys["eta_inv"],
                "PR_base": phys["PR_base"],
                "NOCT": phys["NOCT"],
                "gamma_p": phys["gamma_p"],
                "c_wind": phys["c_wind"],
                "physics_bias": phys["physics_bias"].detach(),
                "residual_max": torch.tensor(self.residual_max),
                "PR_dyn_mean": pr_dyn.mean().detach() if pr_dyn is not None else torch.tensor(float("nan")),
                "PR_dyn_min": pr_dyn.min().detach() if pr_dyn is not None else torch.tensor(float("nan")),
                "PR_dyn_max": pr_dyn.max().detach() if pr_dyn is not None else torch.tensor(float("nan")),
            },
        }
        if pr_dyn is not None:
            out["PR_dyn"] = pr_dyn.unsqueeze(-1)
        return out


In [ ]:
# Pure CNN-BiLSTM-Attention baseline

import torch
import torch.nn as nn
import torch.nn.functional as F

class PureCNNiLSTM(nn.Module):
    def __init__(self, num_features, y_log_mean, y_log_std,
                 cnn_channels=64, kernel_size=3, lstm_hidden=64, lstm_layers=1, seq_dropout=0.10):
        super().__init__()
        self.register_buffer("y_log_mean", torch.tensor([float(y_log_mean)], dtype=torch.float32))
        self.register_buffer("y_log_std",  torch.tensor([float(y_log_std)],  dtype=torch.float32))

        self.encoder = CNNBiLSTMAttnEncoder(
            in_dim=num_features,
            cnn_channels=cnn_channels,
            kernel_size=kernel_size,
            lstm_hidden=lstm_hidden,
            lstm_layers=lstm_layers,
            dropout=seq_dropout,
        )
        ctx_dim = self.encoder.out_dim

        self.head = nn.Sequential(
            nn.Linear(ctx_dim + 2*num_features, 256),
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Dropout(0.10),
            nn.Linear(256, 64),
            nn.SiLU(),
            nn.Linear(64, 1),
        )

        self.calib_a = nn.Parameter(torch.tensor(0.0))
        self.calib_b = nn.Parameter(torch.tensor(1.0))

    def _scaled_to_real(self, y_scaled):
        y_log = y_scaled * self.y_log_std + self.y_log_mean
        return torch.expm1(y_log)

    def forward(self, X, month_idx=None):
        x_last = X[:, -1, :]
        x_mean = X.mean(dim=1)
        ctx, attn = self.encoder(X)

        y_scaled_raw = self.head(torch.cat([x_last, x_mean, ctx], dim=-1))
        y_scaled = self.calib_a + self.calib_b * y_scaled_raw
        y_real = self._scaled_to_real(y_scaled).clamp(min=0.0)

        return {"y_real": y_real, "y_scaled": y_scaled, "attn": attn}


In [ ]:
# Exponential moving average wrapper

import torch
from contextlib import contextmanager

class EMAWrapper:
    def __init__(self, model: torch.nn.Module, decay: float = 0.999):
        self.decay = float(decay)
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: torch.nn.Module):
        d = self.decay
        sd = model.state_dict()
        for k, v in sd.items():
            self.shadow[k].mul_(d).add_(v.detach(), alpha=1.0 - d)

    @torch.no_grad()
    def load_shadow(self, model: torch.nn.Module):
        model.load_state_dict(self.shadow, strict=True)

    def save_shadow(self) -> dict:
        """Return a deep copy of the current shadow state (best checkpoint snapshot)."""
        return {k: v.detach().clone() for k, v in self.shadow.items()}

    def restore_shadow(self, saved: dict):
        """Overwrite ema.shadow with a previously saved snapshot."""
        for k, v in saved.items():
            self.shadow[k].copy_(v)

    @contextmanager
    def apply_to_model(self, model: torch.nn.Module):
        """
        Context manager that temporarily swaps model weights to EMA weights, then restores.
        """
        backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
        self.load_shadow(model)
        try:
            yield model
        finally:
            model.load_state_dict(backup, strict=True)


In [ ]:
# Gated mixture-of-experts model

import torch
import torch.nn as nn
import torch.nn.functional as F

class GatedMoEHybrid(nn.Module):
    """
    y = alpha * y_nn + (1-alpha) * y_phys
    alpha learned from [x_last, x_mean, ctx, phys_feats]
    """
    def __init__(self, num_features, idx_h, idx_tamb, x_means, x_scales, y_log_mean, y_log_std,
                 idx_wind=None, idx_solarrad=None,
                 cnn_channels=64, kernel_size=3, lstm_hidden=64, lstm_layers=1, seq_dropout=0.10):
        super().__init__()
        self.register_buffer("y_log_mean", torch.tensor([float(y_log_mean)], dtype=torch.float32))
        self.register_buffer("y_log_std",  torch.tensor([float(y_log_std)],  dtype=torch.float32))

        self.physics = PhysicsPVDailyMeaningfulMonthlyPR(
            idx_h=idx_h, idx_tamb=idx_tamb, x_means=x_means, x_scales=x_scales,
            idx_wind=idx_wind, idx_solarrad=idx_solarrad,
            pr_month_init=None, trainable=True
        )

        self.encoder = CNNBiLSTMAttnEncoder(
            in_dim=num_features, cnn_channels=cnn_channels, kernel_size=kernel_size,
            lstm_hidden=lstm_hidden, lstm_layers=lstm_layers, dropout=seq_dropout
        )
        ctx_dim = self.encoder.out_dim

        self.nn_head = nn.Sequential(
            nn.Linear(ctx_dim + 2*num_features, 256),
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Dropout(0.10),
            nn.Linear(256, 64),
            nn.SiLU(),
            nn.Linear(64, 1),
        )

        self.gate = nn.Sequential(
            nn.Linear(ctx_dim + 2*num_features + 4, 128),
            nn.SiLU(),
            nn.Linear(128, 1),
        )

        self.calib_a = nn.Parameter(torch.tensor(0.0))
        self.calib_b = nn.Parameter(torch.tensor(1.0))

    def _scaled_to_real(self, y_scaled):
        y_log = y_scaled * self.y_log_std + self.y_log_mean
        return torch.expm1(y_log)

    def _real_to_scaled(self, y_real):
        y_log = torch.log1p(y_real.clamp(min=0.0))
        return (y_log - self.y_log_mean) / (self.y_log_std + 1e-12)

    def forward(self, X, month_idx=None):
        x_last = X[:, -1, :]
        x_mean = X.mean(dim=1)

        phys = self.physics(X, pr_dyn=None, month_idx=month_idx)
        y_phys = phys["E_ac"]                # [N,1]
        y_phys_scaled = self._real_to_scaled(y_phys)

        ctx, attn = self.encoder(X)
        y_nn_scaled_raw = self.nn_head(torch.cat([x_last, x_mean, ctx], dim=-1))
        y_nn_scaled = self.calib_a + self.calib_b * y_nn_scaled_raw
        y_nn = self._scaled_to_real(y_nn_scaled).clamp(min=0.0)

        E_dc = phys["E_dc"] / 60.0
        fT   = phys["f_T"] - 1.0
        PR   = phys["PR_total"] - 0.85
        Tc   = (phys["T_cell"] - 25.0) / 20.0
        phys4 = torch.cat([E_dc, fT, PR, Tc], dim=-1)   # [N,4]

        a_raw = self.gate(torch.cat([x_last, x_mean, ctx, phys4], dim=-1))
        alpha = torch.sigmoid(a_raw)                    # [N,1]

        y = alpha * y_nn + (1.0 - alpha) * y_phys

        return {"y_real": y, "alpha": alpha, "y_nn": y_nn, "y_phys": y_phys, "attn": attn}


In [ ]:
# Latent physical regularization model

class LatentPhysRegModel(nn.Module):
    def __init__(self, num_features, idx_h, idx_tamb, x_means, x_scales, y_log_mean, y_log_std,
                 idx_wind=None, idx_solarrad=None,
                 cnn_channels=64, kernel_size=3, lstm_hidden=64, lstm_layers=1, seq_dropout=0.10):
        super().__init__()
        self.register_buffer("y_log_mean", torch.tensor([float(y_log_mean)], dtype=torch.float32))
        self.register_buffer("y_log_std",  torch.tensor([float(y_log_std)],  dtype=torch.float32))

        self.physics = PhysicsPVDailyMeaningfulMonthlyPR(
            idx_h=idx_h, idx_tamb=idx_tamb, x_means=x_means, x_scales=x_scales,
            idx_wind=idx_wind, idx_solarrad=idx_solarrad,
            pr_month_init=None, trainable=True
        )

        self.encoder = CNNBiLSTMAttnEncoder(
            in_dim=num_features, cnn_channels=cnn_channels, kernel_size=kernel_size,
            lstm_hidden=lstm_hidden, lstm_layers=lstm_layers, dropout=seq_dropout
        )
        ctx_dim = self.encoder.out_dim

        self.head_y = nn.Sequential(
            nn.Linear(ctx_dim + 2*num_features, 256),
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Dropout(0.10),
            nn.Linear(256, 1),
        )

        self.head_phys = nn.Sequential(
            nn.Linear(ctx_dim + 2*num_features, 128),
            nn.SiLU(),
            nn.Linear(128, 2),  # [Tcell_hat_norm, PR_hat_norm]
        )

    def _scaled_to_real(self, y_scaled):
        y_log = y_scaled * self.y_log_std + self.y_log_mean
        return torch.expm1(y_log)

    def forward(self, X, month_idx=None):
        x_last = X[:, -1, :]
        x_mean = X.mean(dim=1)
        ctx, attn = self.encoder(X)

        feats = torch.cat([x_last, x_mean, ctx], dim=-1)

        y_scaled = self.head_y(feats)
        y_real = self._scaled_to_real(y_scaled).clamp(min=0.0)

        phys = self.physics(X, pr_dyn=None, month_idx=month_idx)

        phys_lat = self.head_phys(feats)  # [N,2]: [T_cell_hat_norm, PR_hat_norm]

        T_cell_target = (phys["T_cell"].detach() - 25.0) / 20.0   # [N,1]
        PR_target      = (phys["PR_total"].detach() - 0.85)        # [N,1]
        phys_targets   = torch.cat([T_cell_target, PR_target], dim=-1)  # [N,2]

        return {
            "y_real": y_real,
            "y_scaled": y_scaled,
            "phys_lat": phys_lat,
            "phys_targets": phys_targets,
            "E_ac": phys["E_ac"],
            "attn": attn,
        }


In [ ]:
# Weather calibration neural model

import torch
import torch.nn as nn

class WeatherCalibNN_v4(nn.Module):
    def __init__(self, num_features, idx_h, idx_cloud, y_log_mean, y_log_std,
                 cnn_channels=64, kernel_size=3, lstm_hidden=64, lstm_layers=1, seq_dropout=0.10):
        super().__init__()
        self.register_buffer("y_log_mean", torch.tensor([float(y_log_mean)], dtype=torch.float32))
        self.register_buffer("y_log_std",  torch.tensor([float(y_log_std)],  dtype=torch.float32))

        self.idx_h = int(idx_h)
        self.idx_cloud = int(idx_cloud)
        self.num_features = int(num_features)

        self.calib = nn.Sequential(
            nn.Linear(2 * num_features, 128),
            nn.SiLU(),
            nn.Linear(128, 3),
        )

        self.encoder = CNNBiLSTMAttnEncoder(
            in_dim=num_features,
            cnn_channels=cnn_channels,
            kernel_size=kernel_size,
            lstm_hidden=lstm_hidden,
            lstm_layers=lstm_layers,
            dropout=seq_dropout
        )
        ctx_dim = self.encoder.out_dim

        self.head = nn.Sequential(
            nn.Linear(ctx_dim + 2 * num_features, 256),
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Dropout(0.10),
            nn.Linear(256, 1),
        )

    def _scaled_to_real(self, y_scaled):
        y_log = y_scaled * self.y_log_std + self.y_log_mean
        return torch.expm1(y_log)

    def forward(self, X, month_idx=None):
        N, S, F = X.shape
        assert F == self.num_features, f"Model expects F={self.num_features} but got F={F}"
        assert 0 <= self.idx_h < F
        assert 0 <= self.idx_cloud < F
        assert self.idx_h != self.idx_cloud

        x_last_orig = X[:, -1, :]
        x_mean_orig = X.mean(dim=1)

        p = self.calib(torch.cat([x_last_orig, x_mean_orig], dim=-1))
        sH = 0.90 + 0.20 * torch.sigmoid(p[:, 0:1])   # [N,1] mult. correction: solarenergy in [0.90, 1.10]
        bH = 0.10 * torch.tanh(p[:, 1:2])             # [N,1] add.  correction: solarenergy in [-0.10, +0.10]
        dc = 0.20 * torch.tanh(p[:, 2:3])             # [N,1] add.  correction: cloudcover  in [-0.20, +0.20]

        Xc = X.clone()
        Xc[:, :, self.idx_h:self.idx_h+1] = (
            X[:, :, self.idx_h:self.idx_h+1] * sH.unsqueeze(1) + bH.unsqueeze(1)
        )
        Xc[:, :, self.idx_cloud:self.idx_cloud+1] = (
            X[:, :, self.idx_cloud:self.idx_cloud+1] + dc.unsqueeze(1)
        )

        ctx, attn = self.encoder(Xc)

        x_last_c = Xc[:, -1, :]
        x_mean_c = Xc.mean(dim=1)

        y_scaled = self.head(torch.cat([x_last_c, x_mean_c, ctx], dim=-1))
        y_real = self._scaled_to_real(y_scaled).clamp(min=0.0)

        return {
            "y_real": y_real,
            "y_scaled": y_scaled,
            "calib_params": {"sH": sH.detach(), "bH": bH.detach(), "dc": dc.detach()},
            "attn": attn,
        }


In [ ]:
# Persistence D-1 baseline utilities

import torch
import numpy as np

@torch.no_grad()
def persistence_predict_real_from_X(model_ref, X_seg):
    """
    D-1 persistence using the last available generated-energy feature in the input sequence.

    Assumes:
    - the target variable was log1p + standardized using model_ref.y_log_mean / y_log_std
    - one of the sequence features contains the past generated energy signal
    - preferred feature names:
        'Generated'
        'Generated_ewm3'
        'Generated_ewm7'
    Priority is given to raw 'Generated' if present.

    Returns:
    - y_pred_real: torch tensor on CPU, shape (N,)
    """
    assert "base_features" in globals(), "base_features not found in globals()"
    assert hasattr(model_ref, "y_log_mean") and hasattr(model_ref, "y_log_std"), "model_ref missing y scaling attributes"

    candidate_order = ["Generated", "Generated_ewm3", "Generated_ewm7"]
    feat_name = None
    for c in candidate_order:
        if c in base_features:
            feat_name = c
            break

    if feat_name is None:
        raise ValueError(
            "Persistence baseline could not find any of these features in base_features: "
            "['Generated', 'Generated_ewm3', 'Generated_ewm7']"
        )

    idx_gen = base_features.index(feat_name)

    dev = model_ref.y_log_mean.device if isinstance(model_ref.y_log_mean, torch.Tensor) else X_seg.device

    y_prev_scaled = X_seg[:, -1, idx_gen].to(dev)

    y_log_mean = model_ref.y_log_mean if isinstance(model_ref.y_log_mean, torch.Tensor) else torch.tensor(model_ref.y_log_mean, device=dev)
    y_log_std  = model_ref.y_log_std  if isinstance(model_ref.y_log_std,  torch.Tensor) else torch.tensor(model_ref.y_log_std,  device=dev)

    y_prev_log = y_prev_scaled * y_log_std + y_log_mean
    y_pred_real = torch.expm1(y_prev_log).float()

    return y_pred_real.detach().cpu()

@torch.no_grad()
def eval_persistence_main_pred(
    model_ref,
    X_main, y_main,
    X_pred, y_pred,
    lambda_score=1.0,
    name="Persistence D-1"
):
    """
    Evaluate persistence on MAIN_TEST and PRED_TEST separately.
    model_ref is used only for inverse-transform parameters.
    """
    y_true_main = scaled_to_real_torch(y_main, model_ref, to_cpu=True).squeeze(-1).detach()
    y_hat_main  = persistence_predict_real_from_X(model_ref, X_main)

    y_true_pred = scaled_to_real_torch(y_pred, model_ref, to_cpu=True).squeeze(-1).detach()
    y_hat_pred  = persistence_predict_real_from_X(model_ref, X_pred)

    m_main = compute_metrics(y_true_main, y_hat_main)
    m_pred = compute_metrics(y_true_pred, y_hat_pred)

    d_rmse = m_pred["RMSE"] - m_main["RMSE"]
    score  = m_main["RMSE"] + float(lambda_score) * m_pred["RMSE"]

    print(f"\n=== {name} ===")
    print(f"MAIN  (N={len(X_main)}) | RMSE={m_main['RMSE']:.4f} | MAE={m_main['MAE']:.4f} | R2={m_main['R2']:.4f} | MAPE>1={m_main['MAPE>1']:.3f}%")
    print(f"PRED  (N={len(X_pred)}) | RMSE={m_pred['RMSE']:.4f} | MAE={m_pred['MAE']:.4f} | R2={m_pred['R2']:.4f} | MAPE>1={m_pred['MAPE>1']:.3f}%")
    print(f"ΔRMSE (pred-main) = {d_rmse:+.4f}")
    print(f"SCORE (RMSE_main + {lambda_score}*RMSE_pred) = {score:.4f}")

    return {
        "metrics_main": m_main,
        "metrics_pred": m_pred,
        "dRMSE": d_rmse,
        "score": score,
        "y_true_main": y_true_main,
        "y_pred_main": y_hat_main,
        "y_true_pred": y_true_pred,
        "y_pred_pred": y_hat_pred,
    }

@torch.no_grad()
def rolling_eval_persistence(
    model_ref,
    X_seg, y_seg,
    win=15,
    step=1,
    label="SEGMENT"
):
    """
    Rolling evaluation for persistence on one segment only.
    """
    N = int(X_seg.shape[0])
    assert N >= win, f"{label} too short N={N} for win={win}"

    y_true = scaled_to_real_torch(y_seg, model_ref, to_cpu=True).squeeze(-1).detach()
    y_hat  = persistence_predict_real_from_X(model_ref, X_seg)

    rmses, maes, r2s = [], [], []
    starts = list(range(0, N - win + 1, step))

    for s in starts:
        e = s + win
        m = compute_metrics(y_true[s:e], y_hat[s:e])
        rmses.append(m["RMSE"])
        maes.append(m["MAE"])
        r2s.append(m["R2"])

    rmses = np.array(rmses, dtype=np.float64)
    maes  = np.array(maes, dtype=np.float64)
    r2s   = np.array(r2s, dtype=np.float64)

    summary = {
        "count": int(len(rmses)),
        "rmse_mean": float(np.mean(rmses)),
        "rmse_median": float(np.median(rmses)),
        "rmse_p90": float(np.quantile(rmses, 0.90)),
        "rmse_p10": float(np.quantile(rmses, 0.10)),
        "mae_mean": float(np.mean(maes)),
        "r2_mean": float(np.mean(r2s)),
    }

    print(f"\n--- Persistence rolling {label} summary ---")
    print(f"windows={summary['count']} | RMSE mean={summary['rmse_mean']:.4f} | median={summary['rmse_median']:.4f} | p90={summary['rmse_p90']:.4f}")
    print(f"MAE mean={summary['mae_mean']:.4f} | R2 mean={summary['r2_mean']:.4f}")

    return {
        "starts": np.array(starts),
        "rmse": rmses,
        "mae": maes,
        "r2": r2s,
        "summary": summary
    }


In [ ]:
# Reproducibility setup

import os
import random
import numpy as np
import pandas as pd
import torch

SEEDS = [11, 22, 33, 44, 55]

def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass

    return seed

print("Seeds:", SEEDS)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Shared multi-run utilities

import copy
import torch
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fixed random seeds are used for reproducibility.
USE_AMP_MULTI = False

def make_train_loader(batch_size: int, seed: int):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor, month_train_tensor),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
        generator=g,
        pin_memory=(device.type == "cuda"),
    )

def make_val_loader(batch_size: int = 4096):
    return DataLoader(
        TensorDataset(X_val_tensor, y_val_tensor, month_val_tensor),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        pin_memory=(device.type == "cuda"),
    )

def summarize_runs(run_dicts, model_name):
    rows = []
    for i, d in enumerate(run_dicts):
        row = {"Model": model_name, "Run": i + 1, "Seed": d["seed"]}
        for k, v in d.items():
            if k not in ["seed", "rolling_main", "rolling_pred", "phys_results"]:
                row[k] = v
        rows.append(row)

    df_runs = pd.DataFrame(rows)

    metric_cols = [c for c in df_runs.columns if c not in ["Model", "Run", "Seed"]]
    summary = {"Model": model_name}
    for c in metric_cols:
        summary[f"{c}_mean"] = float(df_runs[c].mean())
        summary[f"{c}_std"] = float(df_runs[c].std(ddof=1)) if len(df_runs) > 1 else 0.0

    return df_runs, pd.DataFrame([summary])

def pick_representative_seed(run_dicts, key="score"):
    vals = np.array([d[key] for d in run_dicts], dtype=float)
    med = np.median(vals)
    idx = int(np.argmin(np.abs(vals - med)))
    return run_dicts[idx]["seed"], idx

def pack_result_dict(seed, res, roll_main, roll_pred):
    return {
        "seed": seed,
        "rmse_main": res["metrics_main"]["RMSE"],
        "mae_main":  res["metrics_main"]["MAE"],
        "r2_main":   res["metrics_main"]["R2"],
        "mape_main": res["metrics_main"]["MAPE>1"],

        "rmse_pred": res["metrics_pred"]["RMSE"],
        "mae_pred":  res["metrics_pred"]["MAE"],
        "r2_pred":   res["metrics_pred"]["R2"],
        "mape_pred": res["metrics_pred"]["MAPE>1"],

        "d_rmse":    res["dRMSE"],
        "score":     res["score"],

        "rolling_main": roll_main["summary"],
        "rolling_pred": roll_pred["summary"],
    }

def result_row_from_summary(summary_df):
    s = summary_df.iloc[0].to_dict()
    return {
        "Model": s["Model"],
        "RMSE_main": s["rmse_main_mean"],
        "RMSE_main_std": s["rmse_main_std"],
        "MAE_main": s["mae_main_mean"],
        "MAE_main_std": s["mae_main_std"],
        "R2_main": s["r2_main_mean"],
        "R2_main_std": s["r2_main_std"],
        "MAPE_main": s["mape_main_mean"],
        "MAPE_main_std": s["mape_main_std"],

        "RMSE_pred": s["rmse_pred_mean"],
        "RMSE_pred_std": s["rmse_pred_std"],
        "MAE_pred": s["mae_pred_mean"],
        "MAE_pred_std": s["mae_pred_std"],
        "R2_pred": s["r2_pred_mean"],
        "R2_pred_std": s["r2_pred_std"],
        "MAPE_pred": s["mape_pred_mean"],
        "MAPE_pred_std": s["mape_pred_std"],

        "ΔRMSE": s["d_rmse_mean"],
        "ΔRMSE_std": s["d_rmse_std"],
        "Score": s["score_mean"],
        "Score_std": s["score_std"],
    }


In [ ]:
# Model builders

def build_hybrid_model():
    num_features = int(X_train_tensor.shape[2])
    y_log_mean = float(scaler_y.mean_.reshape(-1)[0])
    y_log_std  = float(scaler_y.scale_.reshape(-1)[0])

    idx_h        = base_features.index("solarenergy")
    idx_tamb     = base_features.index("tempmax")
    idx_wind     = base_features.index("windspeed") if ("windspeed" in base_features) else None
    idx_solarrad = base_features.index("solarradiation") if ("solarradiation" in base_features) else None

    mdl = HybridPINNMeaningful(
        num_features=num_features,
        idx_h=idx_h,
        idx_tamb=idx_tamb,
        x_means=feat_means,
        x_scales=feat_scales,
        y_log_mean=y_log_mean,
        y_log_std=y_log_std,
        idx_wind=idx_wind,
        idx_solarrad=idx_solarrad,
        cnn_channels=64,
        kernel_size=3,
        lstm_hidden=64,
        lstm_layers=1,
        seq_dropout=0.10,
        pr_dyn_enabled=True,
        residual_max=0.45,
    ).to(device)
    return mdl

def build_pure_model():
    num_features = int(X_train_tensor.shape[2])
    y_log_mean = float(scaler_y.mean_.reshape(-1)[0])
    y_log_std  = float(scaler_y.scale_.reshape(-1)[0])

    mdl = PureCNNiLSTM(
        num_features=num_features,
        y_log_mean=y_log_mean,
        y_log_std=y_log_std,
        cnn_channels=64,
        kernel_size=3,
        lstm_hidden=64,
        lstm_layers=1,
        seq_dropout=0.10,
    ).to(device)
    return mdl

def build_moe_model():
    num_features = int(X_train_tensor.shape[2])
    y_log_mean = float(scaler_y.mean_.reshape(-1)[0])
    y_log_std  = float(scaler_y.scale_.reshape(-1)[0])

    idx_h    = base_features.index("solarenergy")
    idx_tamb = base_features.index("tempmax")
    idx_wind = base_features.index("windspeed") if "windspeed" in base_features else None

    mdl = GatedMoEHybrid(
        num_features=num_features,
        idx_h=idx_h,
        idx_tamb=idx_tamb,
        x_means=feat_means,
        x_scales=feat_scales,
        y_log_mean=y_log_mean,
        y_log_std=y_log_std,
        idx_wind=idx_wind,
    ).to(device)
    return mdl

def build_latent_model():
    num_features = int(X_train_tensor.shape[2])
    y_log_mean = float(scaler_y.mean_.reshape(-1)[0])
    y_log_std  = float(scaler_y.scale_.reshape(-1)[0])

    idx_h    = base_features.index("solarenergy")
    idx_tamb = base_features.index("tempmax")
    idx_wind = base_features.index("windspeed") if "windspeed" in base_features else None

    mdl = LatentPhysRegModel(
        num_features=num_features,
        idx_h=idx_h,
        idx_tamb=idx_tamb,
        x_means=feat_means,
        x_scales=feat_scales,
        y_log_mean=y_log_mean,
        y_log_std=y_log_std,
        idx_wind=idx_wind,
    ).to(device)
    return mdl

def build_weathercalib_model():
    num_features = int(X_train_tensor.shape[2])
    y_log_mean = float(scaler_y.mean_.reshape(-1)[0])
    y_log_std  = float(scaler_y.scale_.reshape(-1)[0])

    idx_h = base_features.index("solarenergy")
    idx_cloud = base_features.index("cloudcover")

    mdl = WeatherCalibNN_v4(
        num_features=num_features,
        idx_h=idx_h,
        idx_cloud=idx_cloud,
        y_log_mean=y_log_mean,
        y_log_std=y_log_std
    ).to(device)
    return mdl


In [ ]:
# Run pure baseline across seeds

import time
def run_pure_once(seed: int):
    set_seed(seed)
    torch.cuda.empty_cache()
    if device.type == "cuda":
        torch.cuda.synchronize()
    run_wall_t0 = time.perf_counter()

    baseline = build_pure_model()
    ema_pure = EMAWrapper(baseline, decay=0.999)

    train_dl = make_train_loader(batch_size=768, seed=seed)
    optim = torch.optim.AdamW(baseline.parameters(), lr=3e-4, weight_decay=1e-3, betas=(0.9, 0.99))
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, mode="min", factor=0.5, patience=12, min_lr=1e-5)

    patience = 80
    best_rmse = float("inf")
    best_epoch = None
    best_ema_shadow_ema_pure = None
    best_model_state_ema_pure = None
    no_imp = 0
    epoch = 0
    stop_reason = None

    CLAMP_EPS = 1e-3
    LOW_W_COEF = 0.12

    def safe_log(x, eps=CLAMP_EPS):
        return torch.log(torch.clamp(x, min=eps))

    def w_log_now(epoch):
        LOG_START, LOG_END, LOG_TARGET = 0, 120, 0.35
        if epoch < LOG_START:
            return 0.0
        if epoch >= LOG_END:
            return float(LOG_TARGET)
        t = (epoch - LOG_START) / max(1, (LOG_END - LOG_START))
        return float(LOG_TARGET * t)

    if device.type == "cuda":
        torch.cuda.synchronize()
    train_wall_t0 = time.perf_counter()

    while True:
        baseline.train()
        use_amp_local = USE_AMP_MULTI and (epoch >= 10)
        w_log_t = w_log_now(epoch)

        for xb, yb, mb in train_dl:
            xb = xb.to(device)
            yb = yb.to(device)

            optim.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda" if device.type == "cuda" else "cpu", enabled=use_amp_local):
                out = baseline(xb)
                y_pred_real = out["y_real"].squeeze(-1)
                y_true_real = scaled_to_real_torch(yb, baseline).squeeze(-1)

                w_low = 1.0 + LOW_W_COEF / (1.0 + torch.clamp(y_true_real, min=0.0))
                loss_kwh_i = F.smooth_l1_loss(y_pred_real, y_true_real, reduction="none")
                loss_kwh = torch.mean(w_low * loss_kwh_i)

                loss_log = F.smooth_l1_loss(
                    safe_log(torch.clamp(y_pred_real, min=0.0)),
                    safe_log(torch.clamp(y_true_real, min=0.0))
                )

                loss = (1.0 - w_log_t) * loss_kwh + w_log_t * loss_log

            loss.backward()
            torch.nn.utils.clip_grad_norm_(baseline.parameters(), max_norm=1.0)
            optim.step()
            ema_pure.update(baseline)

        with torch.no_grad():
            y_pred_val = model_predict_real(baseline, X_val_tensor, month_idx=None, use_ema=True, ema=ema_pure)
            y_true_val = scaled_to_real_torch(y_val_tensor, baseline, to_cpu=True).squeeze(-1).detach()
            rmse_val = compute_metrics(y_true_val, y_pred_val)["RMSE"]

        sched.step(rmse_val)

        if rmse_val < best_rmse - 0.002:
            best_rmse = rmse_val
            best_epoch = epoch + 1
            best_ema_shadow_ema_pure = ema_pure.save_shadow()
            best_model_state_ema_pure = {k: v.detach().cpu().clone() for k, v in baseline.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1

        if no_imp >= patience or float(optim.param_groups[0]["lr"]) <= 1e-5:
            stop_reason = "patience" if no_imp >= patience else "min_lr"
            break

        epoch += 1

    if device.type == "cuda":
        torch.cuda.synchronize()
    train_wall_seconds = time.perf_counter() - train_wall_t0

    if best_ema_shadow_ema_pure is not None:
        baseline.load_state_dict(best_model_state_ema_pure, strict=True)
        ema_pure.restore_shadow(best_ema_shadow_ema_pure)

    res = eval_main_pred_score(
        baseline,
        X_main_test_tensor, y_main_test_tensor, month_main_test_tensor,
        X_pred_test_tensor, y_pred_test_true_tensor, month_pred_test_tensor,
        name=f"PURE CNN-BiLSTM-attn | seed={seed}",
        lambda_score=1.0,
        use_ema=True,
        ema=ema_pure,
        also_phys=False
    )

    roll_main = rolling_eval(
        baseline, X_main_test_tensor, y_main_test_tensor, month_main_test_tensor,
        win=15, step=1, use_ema=True, ema=ema_pure, label="MAIN_TEST"
    )
    roll_pred = rolling_eval(
        baseline, X_pred_test_tensor, y_pred_test_true_tensor, month_pred_test_tensor,
        win=15, step=1, use_ema=True, ema=ema_pure, label="PRED_TEST"
    )

    out = pack_result_dict(seed, res, roll_main, roll_pred)
    if device.type == "cuda":
        torch.cuda.synchronize()
    run_wall_seconds = time.perf_counter() - run_wall_t0

    train_meta = {
        "Model": "Pure CNN-BiLSTM-Attn",
        "seed": int(seed),
        "best_epoch": int(best_epoch) if best_epoch is not None else None,
        "epochs_ran": int(epoch + 1),
        "best_val_rmse": float(best_rmse),
        "train_wall_clock_seconds": float(train_wall_seconds),
        "run_wall_clock_seconds": float(run_wall_seconds),
        "stop_reason": stop_reason or "completed",
    }
    return out, baseline, ema_pure, train_meta

pure_runs = []
pure_artifacts = {}

for s in SEEDS:
    print("\n" + "="*80)
    print(f"PURE RUN | seed={s}")
    print("="*80)
    out, mdl, ema, train_meta = run_pure_once(s)
    pure_runs.append(out)
    pure_artifacts[s] = {"model": mdl, "ema": ema, "train_meta": train_meta}

df_pure_runs, df_pure_summary = summarize_runs(pure_runs, "PURE CNN-BiLSTM-attn")
display(df_pure_runs)
display(df_pure_summary)


In [ ]:
# Run hybrid model across seeds

import time
def run_hybrid_once(seed: int):
    set_seed(seed)
    torch.cuda.empty_cache()
    if device.type == "cuda":
        torch.cuda.synchronize()
    run_wall_t0 = time.perf_counter()

    model = build_hybrid_model()
    ema_hyb = EMAWrapper(model, decay=0.999)

    train_dl = make_train_loader(batch_size=768, seed=seed)
    optim = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3, betas=(0.9, 0.99))
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, mode="min", factor=0.5, patience=12, min_lr=1e-5)

    CLAMP_EPS = 1e-3
    def safe_log(x, eps=CLAMP_EPS):
        return torch.log(torch.clamp(x, min=eps))

    CAP_START, CAP_END = 30, 90
    CAP_TARGET = 0.05
    LOG_START, LOG_END, LOG_TARGET = 0, 120, 0.35
    TAIL_START, TAIL_SPAN, TAIL_MIN = 500, 250, 0.80

    def ramp(epoch, start, end, target):
        if epoch < start: return 0.0
        if epoch >= end: return float(target)
        t = (epoch - start) / max(1, (end - start))
        return float(target * t)

    def cap_ratio_now(epoch): return ramp(epoch, CAP_START, CAP_END, CAP_TARGET)
    def w_eff_now(epoch, base): return ramp(epoch, CAP_START, CAP_END, base)
    def w_log_now(epoch, base): return ramp(epoch, LOG_START, LOG_END, base)

    def tail_scale(epoch):
        if epoch < TAIL_START:
            return 1.0
        t = (epoch - TAIL_START) / max(1, TAIL_SPAN)
        t = float(max(0.0, min(1.0, t)))
        return float(1.0 - (1.0 - TAIL_MIN) * t)

    w_dyn_base = 0.015
    w_eff_base = 0.12
    w_log_base = 0.35

    ADAPT_EVERY = 5
    DYNA_MAX = 0.045
    EFF_MAX  = 0.20
    LOG_MAX  = 0.45

    lambda_prior = 2e-4
    PRDYN_PRIOR_W = 0.05
    LOW_W_COEF = 0.12

    CYCLE_LEN = 80
    UNFREEZE_FOR = 12
    physics_core_frozen = False

    def set_physics_trainable(model, trainable: bool):
        for n, p in model.physics.named_parameters():
            if ("PR_base" in n) or ("pr_month" in n) or ("physics_bias" in n):
                p.requires_grad = True
            else:
                p.requires_grad = bool(trainable)

    set_physics_trainable(model, trainable=True)

    patience = 90
    best_rmse = float("inf")
    best_epoch = None
    best_ema_shadow_ema_hyb = None
    best_model_state_ema_hyb = None
    no_imp = 0
    epoch = 0
    stop_reason = None

    if device.type == "cuda":
        torch.cuda.synchronize()
    train_wall_t0 = time.perf_counter()

    while True:
        model.train()

        pos_in_cycle = epoch % CYCLE_LEN
        if pos_in_cycle == 0 and epoch > 0:
            set_physics_trainable(model, trainable=False)
            physics_core_frozen = True

        if (pos_in_cycle >= (CYCLE_LEN - UNFREEZE_FOR)) and physics_core_frozen:
            set_physics_trainable(model, trainable=True)
            physics_core_frozen = False

        cap_ratio_t = cap_ratio_now(epoch)
        tail = tail_scale(epoch)
        w_eff_t = w_eff_now(epoch, w_eff_base) * tail
        w_log_t = w_log_now(epoch, w_log_base) * tail
        w_dyn_t = w_dyn_base

        for xb, yb, mb in train_dl:
            xb = xb.to(device)
            yb = yb.to(device)
            mb = mb.to(device).view(-1)

            optim.zero_grad(set_to_none=True)

            out = model(xb, month_idx=mb)

            y_pred_real = out["y_real"].squeeze(-1)
            y_true_real = scaled_to_real_torch(yb, model).squeeze(-1)
            E_ac = out["E_ac"].squeeze(-1)

            w_low = 1.0 + LOW_W_COEF / (1.0 + torch.clamp(y_true_real, min=0.0))
            loss_kwh_i = F.smooth_l1_loss(y_pred_real, y_true_real, reduction="none")
            loss_kwh = torch.mean(w_low * loss_kwh_i)

            loss_log = F.smooth_l1_loss(
                safe_log(torch.clamp(y_pred_real, min=0.0)),
                safe_log(torch.clamp(y_true_real, min=0.0))
            )

            y_safe = torch.clamp(y_pred_real, min=CLAMP_EPS)
            e_safe = torch.clamp(E_ac, min=CLAMP_EPS)
            d = safe_log(y_safe) - safe_log(e_safe)

            w_up = 1.0
            w_dn = 0.25
            loss_dyn = torch.mean(torch.where(d > 0, w_up * d*d, w_dn * d*d))

            cap_ref = (1.0 + cap_ratio_t) * E_ac.detach()
            over_pos = torch.clamp(y_pred_real - cap_ref, min=0.0)
            loss_eff = F.smooth_l1_loss(over_pos, torch.zeros_like(over_pos))

            loss_prior = lambda_prior * (
                0.02  * (model.physics.A_total - 65.0).pow(2) +
                5.0   * (model.physics.eta_ref - 0.15).pow(2) +
                2.0   * (model.physics.PR_base - 0.82).pow(2) +
                5.0   * (model.physics.eta_inv - 0.97).pow(2) +
                0.10  * (model.physics.NOCT - 45.0).pow(2) +
                200.0 * (model.physics.gamma_p + 0.004).pow(2) +
                0.50  * (model.physics.c_wind - 0.7).pow(2) +
                5.0   * (model.physics.physics_bias).pow(2)
            )

            if "PR_dyn" in out:
                pr_dyn = out["PR_dyn"].squeeze(-1)
                loss_prprior = PRDYN_PRIOR_W * torch.mean((pr_dyn - 1.0).pow(2))
            else:
                loss_prprior = torch.tensor(0.0, device=device)

            loss_data = (1.0 - w_log_t) * loss_kwh + w_log_t * loss_log
            loss = loss_data + w_dyn_t * loss_dyn + w_eff_t * loss_eff + loss_prior + loss_prprior

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optim.step()
            ema_hyb.update(model)

        with torch.no_grad():
            y_pred_val = model_predict_real(model, X_val_tensor, month_idx=month_val_tensor, use_ema=True, ema=ema_hyb)
            y_true_val = scaled_to_real_torch(y_val_tensor, model, to_cpu=True).squeeze(-1).detach()
            rmse_val = compute_metrics(y_true_val, y_pred_val)["RMSE"]

        sched.step(rmse_val)

        if rmse_val < best_rmse - 0.002:
            best_rmse = rmse_val
            best_epoch = epoch + 1
            best_ema_shadow_ema_hyb = ema_hyb.save_shadow()
            best_model_state_ema_hyb = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1

        if epoch % ADAPT_EVERY == 0 and epoch > 0:
            if rmse_val > best_rmse - 0.001:
                w_dyn_base = min(DYNA_MAX, w_dyn_base * 1.12)
                w_eff_base = min(EFF_MAX,  w_eff_base * 1.08)
            else:
                w_dyn_base = max(0.010, w_dyn_base * 0.98)
                w_eff_base = max(0.08,  w_eff_base * 0.98)

            w_log_base = max(0.20, min(LOG_MAX, w_log_base))

        if no_imp >= patience or float(optim.param_groups[0]["lr"]) <= 1e-5:
            stop_reason = "patience" if no_imp >= patience else "min_lr"
            break

        epoch += 1

    if device.type == "cuda":
        torch.cuda.synchronize()
    train_wall_seconds = time.perf_counter() - train_wall_t0

    if best_ema_shadow_ema_hyb is not None:
        model.load_state_dict(best_model_state_ema_hyb, strict=True)
        ema_hyb.restore_shadow(best_ema_shadow_ema_hyb)

    res = eval_main_pred_score(
        model,
        X_main_test_tensor, y_main_test_tensor, month_main_test_tensor,
        X_pred_test_tensor, y_pred_test_true_tensor, month_pred_test_tensor,
        name=f"HYBRID Anchor--Residual | seed={seed}",
        lambda_score=1.0,
        use_ema=True,
        ema=ema_hyb,
        also_phys=True
    )

    roll_main = rolling_eval(
        model, X_main_test_tensor, y_main_test_tensor, month_main_test_tensor,
        win=15, step=1, use_ema=True, ema=ema_hyb, label="MAIN_TEST"
    )
    roll_pred = rolling_eval(
        model, X_pred_test_tensor, y_pred_test_true_tensor, month_pred_test_tensor,
        win=15, step=1, use_ema=True, ema=ema_hyb, label="PRED_TEST"
    )

    out = pack_result_dict(seed, res, roll_main, roll_pred)
    out["phys_results"] = {
        "seed": seed,
        "rmse_main": res["phys_main"]["RMSE"],
        "mae_main":  res["phys_main"]["MAE"],
        "r2_main":   res["phys_main"]["R2"],
        "mape_main": res["phys_main"]["MAPE>1"],
        "rmse_pred": res["phys_pred"]["RMSE"],
        "mae_pred":  res["phys_pred"]["MAE"],
        "r2_pred":   res["phys_pred"]["R2"],
        "mape_pred": res["phys_pred"]["MAPE>1"],
        "d_rmse":    res["phys_dRMSE"],
        "score":     res["phys_score"],
    }
    if device.type == "cuda":
        torch.cuda.synchronize()
    run_wall_seconds = time.perf_counter() - run_wall_t0

    train_meta = {
        "Model": "Hybrid Anchor-Residual",
        "seed": int(seed),
        "best_epoch": int(best_epoch) if best_epoch is not None else None,
        "epochs_ran": int(epoch + 1),
        "best_val_rmse": float(best_rmse),
        "train_wall_clock_seconds": float(train_wall_seconds),
        "run_wall_clock_seconds": float(run_wall_seconds),
        "stop_reason": stop_reason or "completed",
    }
    return out, model, ema_hyb, train_meta

hyb_runs = []
hyb_phys_runs = []
hyb_artifacts = {}

for s in SEEDS:
    print("\n" + "="*80)
    print(f"HYBRID RUN | seed={s}")
    print("="*80)
    out, mdl, ema, train_meta = run_hybrid_once(s)
    hyb_runs.append(out)
    hyb_phys_runs.append(out["phys_results"])
    hyb_artifacts[s] = {"model": mdl, "ema": ema, "train_meta": train_meta}

df_hyb_runs, df_hyb_summary = summarize_runs(hyb_runs, "Hybrid Anchor--Residual")
df_hyb_phys_runs, df_hyb_phys_summary = summarize_runs(hyb_phys_runs, "PHYS-only")
display(df_hyb_runs)
display(df_hyb_summary)
display(df_hyb_phys_runs)
display(df_hyb_phys_summary)


In [ ]:
# Run gated MoE model across seeds

import time
def run_moe_once(seed: int):
    set_seed(seed)
    torch.cuda.empty_cache()
    if device.type == "cuda":
        torch.cuda.synchronize()
    run_wall_t0 = time.perf_counter()

    moe = build_moe_model()
    ema_moe = EMAWrapper(moe, decay=0.999)

    train_dl = make_train_loader(batch_size=768, seed=seed)

    optim = torch.optim.AdamW(moe.parameters(), lr=3e-4, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, mode="min", factor=0.5, patience=12, min_lr=1e-5)

    GATE_REG_W = 0.002
    patience = 60
    best_rmse = float("inf")
    best_epoch = None
    best_ema_shadow_ema_moe = None
    best_model_state_ema_moe = None
    no_imp = 0
    epoch = 0
    stop_reason = None

    if device.type == "cuda":
        torch.cuda.synchronize()
    train_wall_t0 = time.perf_counter()

    while True:
        moe.train()
        for xb, yb, mb in train_dl:
            xb = xb.to(device)
            yb = yb.to(device)
            mb = mb.to(device).view(-1)

            optim.zero_grad(set_to_none=True)

            out = moe(xb, month_idx=mb)
            y_pred = out["y_real"].squeeze(-1)
            y_true = scaled_to_real_torch(yb, moe).squeeze(-1)

            loss = F.smooth_l1_loss(y_pred, y_true)
            a = out["alpha"].squeeze(-1)
            reg = torch.mean((a - 0.5) ** 2)
            loss = loss + GATE_REG_W * reg

            loss.backward()
            torch.nn.utils.clip_grad_norm_(moe.parameters(), 1.0)
            optim.step()
            ema_moe.update(moe)

        with torch.no_grad():
            y_pred_val = model_predict_real(moe, X_val_tensor, month_val_tensor, use_ema=True, ema=ema_moe)
            y_true_val = scaled_to_real_torch(y_val_tensor, moe, to_cpu=True).squeeze(-1).detach()
            rmse_val = compute_metrics(y_true_val, y_pred_val)["RMSE"]

        sched.step(rmse_val)

        if rmse_val < best_rmse - 0.002:
            best_rmse = rmse_val
            best_epoch = epoch + 1
            best_ema_shadow_ema_moe = ema_moe.save_shadow()
            best_model_state_ema_moe = {k: v.detach().cpu().clone() for k, v in moe.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1

        if no_imp >= patience or float(optim.param_groups[0]["lr"]) <= 1e-5:
            stop_reason = "patience" if no_imp >= patience else "min_lr"
            break

        epoch += 1

    if device.type == "cuda":
        torch.cuda.synchronize()
    train_wall_seconds = time.perf_counter() - train_wall_t0

    if best_ema_shadow_ema_moe is not None:
        moe.load_state_dict(best_model_state_ema_moe, strict=True)
        ema_moe.restore_shadow(best_ema_shadow_ema_moe)

    res = eval_main_pred_score(
        moe,
        X_main_test_tensor, y_main_test_tensor, month_main_test_tensor,
        X_pred_test_tensor, y_pred_test_true_tensor, month_pred_test_tensor,
        name=f"MoE-Gated Hybrid | seed={seed}",
        lambda_score=1.0,
        use_ema=True,
        ema=ema_moe,
        also_phys=False
    )

    roll_main = rolling_eval(
        moe, X_main_test_tensor, y_main_test_tensor, month_main_test_tensor,
        win=15, step=1, use_ema=True, ema=ema_moe, label="MAIN_TEST"
    )
    roll_pred = rolling_eval(
        moe, X_pred_test_tensor, y_pred_test_true_tensor, month_pred_test_tensor,
        win=15, step=1, use_ema=True, ema=ema_moe, label="PRED_TEST"
    )

    out = pack_result_dict(seed, res, roll_main, roll_pred)
    if device.type == "cuda":
        torch.cuda.synchronize()
    run_wall_seconds = time.perf_counter() - run_wall_t0

    train_meta = {
        "Model": "MoE-Gated Hybrid",
        "seed": int(seed),
        "best_epoch": int(best_epoch) if best_epoch is not None else None,
        "epochs_ran": int(epoch + 1),
        "best_val_rmse": float(best_rmse),
        "train_wall_clock_seconds": float(train_wall_seconds),
        "run_wall_clock_seconds": float(run_wall_seconds),
        "stop_reason": stop_reason or "completed",
    }
    return out, moe, ema_moe, train_meta

moe_runs = []
moe_artifacts = {}

for s in SEEDS:
    print("\n" + "="*80)
    print(f"MoE RUN | seed={s}")
    print("="*80)
    out, mdl, ema, train_meta = run_moe_once(s)
    moe_runs.append(out)
    moe_artifacts[s] = {"model": mdl, "ema": ema, "train_meta": train_meta}

df_moe_runs, df_moe_summary = summarize_runs(moe_runs, "MoE-Gated Hybrid")
display(df_moe_runs)
display(df_moe_summary)


In [ ]:
# Run latent-regularization model across seeds

import time
def run_latent_once(seed: int):
    set_seed(seed)
    torch.cuda.empty_cache()
    if device.type == "cuda":
        torch.cuda.synchronize()
    run_wall_t0 = time.perf_counter()

    lat = build_latent_model()
    ema_lat = EMAWrapper(lat, decay=0.999)

    train_dl = make_train_loader(batch_size=768, seed=seed)

    optim = torch.optim.AdamW(lat.parameters(), lr=3e-4, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, mode="min", factor=0.5, patience=12, min_lr=1e-5)

    patience = 80
    best_rmse = float("inf")
    best_epoch = None
    best_ema_shadow_ema_lat = None
    best_model_state_ema_lat = None
    no_imp = 0
    epoch = 0
    stop_reason = None

    if device.type == "cuda":
        torch.cuda.synchronize()
    train_wall_t0 = time.perf_counter()

    while True:
        lat.train()
        for xb, yb, mb in train_dl:
            xb = xb.to(device)
            yb = yb.to(device)
            mb = mb.to(device).view(-1)

            optim.zero_grad(set_to_none=True)

            out = lat(xb, month_idx=mb)
            y_pred = out["y_real"].squeeze(-1)
            y_true = scaled_to_real_torch(yb, lat).squeeze(-1)

            loss_data = F.smooth_l1_loss(y_pred, y_true)

            loss_phys_lat = F.mse_loss(out["phys_lat"], out["phys_targets"])

            E_ac = out["E_ac"].squeeze(-1).detach()
            y_safe   = torch.clamp(y_pred, min=1e-3)
            Eac_safe = torch.clamp(E_ac,   min=1e-3)
            loss_anchor = F.smooth_l1_loss(
                torch.log(y_safe), torch.log(Eac_safe)
            )

            loss = loss_data + 0.10 * loss_phys_lat + 0.05 * loss_anchor
            loss.backward()
            torch.nn.utils.clip_grad_norm_(lat.parameters(), 1.0)
            optim.step()
            ema_lat.update(lat)

        with torch.no_grad():
            y_pred_val = model_predict_real(lat, X_val_tensor, month_val_tensor, use_ema=True, ema=ema_lat)
            y_true_val = scaled_to_real_torch(y_val_tensor, lat, to_cpu=True).squeeze(-1).detach()
            rmse_val = compute_metrics(y_true_val, y_pred_val)["RMSE"]

        sched.step(rmse_val)

        if rmse_val < best_rmse - 0.002:
            best_rmse = rmse_val
            best_epoch = epoch + 1
            best_ema_shadow_ema_lat = ema_lat.save_shadow()
            best_model_state_ema_lat = {k: v.detach().cpu().clone() for k, v in lat.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1

        if no_imp >= patience or float(optim.param_groups[0]["lr"]) <= 1e-5:
            stop_reason = "patience" if no_imp >= patience else "min_lr"
            break

        epoch += 1

    if device.type == "cuda":
        torch.cuda.synchronize()
    train_wall_seconds = time.perf_counter() - train_wall_t0

    if best_ema_shadow_ema_lat is not None:
        lat.load_state_dict(best_model_state_ema_lat, strict=True)
        ema_lat.restore_shadow(best_ema_shadow_ema_lat)

    res = eval_main_pred_score(
        lat,
        X_main_test_tensor, y_main_test_tensor, month_main_test_tensor,
        X_pred_test_tensor, y_pred_test_true_tensor, month_pred_test_tensor,
        name=f"Latent Physical Regularization | seed={seed}",
        lambda_score=1.0,
        use_ema=True,
        ema=ema_lat,
        also_phys=False
    )

    roll_main = rolling_eval(
        lat, X_main_test_tensor, y_main_test_tensor, month_main_test_tensor,
        win=15, step=1, use_ema=True, ema=ema_lat, label="MAIN_TEST"
    )
    roll_pred = rolling_eval(
        lat, X_pred_test_tensor, y_pred_test_true_tensor, month_pred_test_tensor,
        win=15, step=1, use_ema=True, ema=ema_lat, label="PRED_TEST"
    )

    out = pack_result_dict(seed, res, roll_main, roll_pred)
    if device.type == "cuda":
        torch.cuda.synchronize()
    run_wall_seconds = time.perf_counter() - run_wall_t0

    train_meta = {
        "Model": "Latent Physical Regularization",
        "seed": int(seed),
        "best_epoch": int(best_epoch) if best_epoch is not None else None,
        "epochs_ran": int(epoch + 1),
        "best_val_rmse": float(best_rmse),
        "train_wall_clock_seconds": float(train_wall_seconds),
        "run_wall_clock_seconds": float(run_wall_seconds),
        "stop_reason": stop_reason or "completed",
    }
    return out, lat, ema_lat, train_meta

lat_runs = []
lat_artifacts = {}

for s in SEEDS:
    print("\n" + "="*80)
    print(f"LATENT RUN | seed={s}")
    print("="*80)
    out, mdl, ema, train_meta = run_latent_once(s)
    lat_runs.append(out)
    lat_artifacts[s] = {"model": mdl, "ema": ema, "train_meta": train_meta}

df_lat_runs, df_lat_summary = summarize_runs(lat_runs, "Latent Physical Regularization")
display(df_lat_runs)
display(df_lat_summary)


In [ ]:
# Run weather-calibration model across seeds

import time
def run_weathercalib_once(seed: int):
    set_seed(seed)
    torch.cuda.empty_cache()
    if device.type == "cuda":
        torch.cuda.synchronize()
    run_wall_t0 = time.perf_counter()

    wc = build_weathercalib_model()
    ema_wc = EMAWrapper(wc, decay=0.999)

    train_dl = make_train_loader(batch_size=768, seed=seed)

    optim = torch.optim.AdamW(wc.parameters(), lr=3e-4, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, mode="min", factor=0.5, patience=12, min_lr=1e-5)

    patience = 80
    best_rmse = float("inf")
    best_epoch = None
    best_ema_shadow_ema_wc = None
    best_model_state_ema_wc = None
    no_imp = 0
    epoch = 0
    stop_reason = None

    if device.type == "cuda":
        torch.cuda.synchronize()
    train_wall_t0 = time.perf_counter()

    while True:
        wc.train()
        for xb, yb, mb in train_dl:
            xb = xb.to(device)
            yb = yb.to(device)

            optim.zero_grad(set_to_none=True)

            out = wc(xb)
            y_pred = out["y_real"].squeeze(-1)
            y_true = scaled_to_real_torch(yb, wc).squeeze(-1)

            loss = F.smooth_l1_loss(y_pred, y_true)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(wc.parameters(), 1.0)
            optim.step()
            ema_wc.update(wc)

        with torch.no_grad():
            y_pred_val = model_predict_real(wc, X_val_tensor, month_idx=None, use_ema=True, ema=ema_wc)
            y_true_val = scaled_to_real_torch(y_val_tensor, wc, to_cpu=True).squeeze(-1).detach()
            rmse_val = compute_metrics(y_true_val, y_pred_val)["RMSE"]

        sched.step(rmse_val)

        if rmse_val < best_rmse - 0.002:
            best_rmse = rmse_val
            best_epoch = epoch + 1
            best_ema_shadow_ema_wc = ema_wc.save_shadow()
            best_model_state_ema_wc = {k: v.detach().cpu().clone() for k, v in wc.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1

        if no_imp >= patience or float(optim.param_groups[0]["lr"]) <= 1e-5:
            stop_reason = "patience" if no_imp >= patience else "min_lr"
            break

        epoch += 1

    if device.type == "cuda":
        torch.cuda.synchronize()
    train_wall_seconds = time.perf_counter() - train_wall_t0

    if best_ema_shadow_ema_wc is not None:
        wc.load_state_dict(best_model_state_ema_wc, strict=True)
        ema_wc.restore_shadow(best_ema_shadow_ema_wc)

    res = eval_main_pred_score(
        wc,
        X_main_test_tensor, y_main_test_tensor, month_main_test_tensor,
        X_pred_test_tensor, y_pred_test_true_tensor, month_pred_test_tensor,
        name=f"WeatherCalib Pre-Net | seed={seed}",
        lambda_score=1.0,
        use_ema=True,
        ema=ema_wc,
        also_phys=False
    )

    roll_main = rolling_eval(
        wc, X_main_test_tensor, y_main_test_tensor, month_main_test_tensor,
        win=15, step=1, use_ema=True, ema=ema_wc, label="MAIN_TEST"
    )
    roll_pred = rolling_eval(
        wc, X_pred_test_tensor, y_pred_test_true_tensor, month_pred_test_tensor,
        win=15, step=1, use_ema=True, ema=ema_wc, label="PRED_TEST"
    )

    out = pack_result_dict(seed, res, roll_main, roll_pred)
    if device.type == "cuda":
        torch.cuda.synchronize()
    run_wall_seconds = time.perf_counter() - run_wall_t0

    train_meta = {
        "Model": "WeatherCalib Pre-Net",
        "seed": int(seed),
        "best_epoch": int(best_epoch) if best_epoch is not None else None,
        "epochs_ran": int(epoch + 1),
        "best_val_rmse": float(best_rmse),
        "train_wall_clock_seconds": float(train_wall_seconds),
        "run_wall_clock_seconds": float(run_wall_seconds),
        "stop_reason": stop_reason or "completed",
    }
    return out, wc, ema_wc, train_meta

wc_runs = []
wc_artifacts = {}

for s in SEEDS:
    print("\n" + "="*80)
    print(f"WEATHERCALIB RUN | seed={s}")
    print("="*80)
    out, mdl, ema, train_meta = run_weathercalib_once(s)
    wc_runs.append(out)
    wc_artifacts[s] = {"model": mdl, "ema": ema, "train_meta": train_meta}

df_wc_runs, df_wc_summary = summarize_runs(wc_runs, "WeatherCalib Pre-Net")
display(df_wc_runs)
display(df_wc_summary)


In [ ]:
# Figure: physics-injection points

import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10.8, 3.4))
ax.axis("off")

def box(x, y, w, h, text, fc="#f7f7f7", ec="#333333", lw=1.3):
    rect = patches.FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.04",
        facecolor=fc, edgecolor=ec, linewidth=lw
    )
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=9)

def arrow(x1, y1, x2, y2, color="#333333", lw=1.4):
    ax.annotate(
        "", xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle="->", color=color, lw=lw, shrinkA=2, shrinkB=2)
    )

box(0.03, 0.45, 0.14, 0.18, "Input sequence\n$X_{t-S+1:t}$", "#e8f1fb")
box(0.23, 0.45, 0.17, 0.18, "Temporal CNN", "#f7f7f7")
box(0.45, 0.45, 0.18, 0.18, "BiLSTM + Attention\ncontext", "#f7f7f7")
box(0.70, 0.45, 0.14, 0.18, "Prediction\nhead", "#f7f7f7")
box(0.90, 0.45, 0.08, 0.18, "$\\hat{y}$", "#e8f1fb")

arrow(0.17, 0.54, 0.23, 0.54)
arrow(0.40, 0.54, 0.45, 0.54)
arrow(0.63, 0.54, 0.70, 0.54)
arrow(0.84, 0.54, 0.90, 0.54)

ax.text(0.49, 0.78, "Common CNN-BiLSTM-Attn backbone", ha="center", fontsize=10, weight="bold")

box(0.12, 0.13, 0.16, 0.16, "Pre-Net\nCalibration", "#fff0d6", "#a26700")
box(0.44, 0.13, 0.18, 0.16, "Latent Physical\nRegularization", "#e7f7e7", "#2e7d32")
box(0.66, 0.13, 0.16, 0.16, "MoE-Gated\nFusion", "#f1e8ff", "#6a3dad")
box(0.79, 0.13, 0.14, 0.16, "Physics\nAnchor", "#fde8e8", "#9d2f2f")

arrow(0.17, 0.45, 0.20, 0.29, "#a26700")
arrow(0.53, 0.45, 0.53, 0.29, "#2e7d32")
arrow(0.74, 0.45, 0.74, 0.29, "#6a3dad")
arrow(0.79, 0.45, 0.86, 0.29, "#9d2f2f")

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
fig.savefig("fig_1_physics_injection_points.pdf", bbox_inches="tight")
fig.savefig("fig_1_physics_injection_points.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: fig_1_physics_injection_points.pdf")
print("Saved: fig_1_physics_injection_points.png")


In [ ]:
# Model parameter counts

import torch
import pandas as pd
import numpy as np

def count_params(model):
    """Count the total and trainable parameters of the model."""
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

model_instances = {
    "Pure CNN-BiLSTM-Attn":          pure_artifacts[SEEDS[0]]["model"],
    "Hybrid Anchor-Residual":        hyb_artifacts[SEEDS[0]]["model"],
    "MoE-Gated Hybrid":              moe_artifacts[SEEDS[0]]["model"],
    "Latent Physical Regularization":lat_artifacts[SEEDS[0]]["model"],
    "WeatherCalib Pre-Net":          wc_artifacts[SEEDS[0]]["model"],
}

param_rows = []
for name, mdl in model_instances.items():
    tot, tr = count_params(mdl)
    param_rows.append({"Model": name, "Total params": tot, "Trainable params": tr})

df_params = pd.DataFrame(param_rows)
print("=== Parameter Counts ===")
display(df_params)


In [ ]:
# Training wall-clock time summary

import pandas as pd
import numpy as np

def collect_training_metadata(artifact_dict):
    rows = []
    for seed, artifact in artifact_dict.items():
        meta = artifact.get("train_meta", {}).copy()
        meta.setdefault("seed", seed)
        rows.append(meta)
    return pd.DataFrame(rows)

artifact_by_model = {
    "Pure CNN-BiLSTM-Attn":           pure_artifacts,
    "Hybrid Anchor-Residual":         hyb_artifacts,
    "MoE-Gated Hybrid":               moe_artifacts,
    "Latent Physical Regularization": lat_artifacts,
    "WeatherCalib Pre-Net":           wc_artifacts,
}

metadata_frames = []
timing_rows = []

for name, artifacts in artifact_by_model.items():
    meta_df = collect_training_metadata(artifacts)
    required_cols = {
        "best_epoch",
        "epochs_ran",
        "best_val_rmse",
        "train_wall_clock_seconds",
        "run_wall_clock_seconds",
    }
    missing = required_cols.difference(meta_df.columns)
    if missing:
        raise ValueError(
            f"Missing real wall-clock timing columns for {name}: {sorted(missing)}. "
            "Re-run the five-seed training cells in this FINAL notebook before COST-2."
        )

    metadata_frames.append(meta_df)

    best_epochs = pd.to_numeric(meta_df["best_epoch"], errors="coerce")
    epochs_ran = pd.to_numeric(meta_df["epochs_ran"], errors="coerce")
    train_times = pd.to_numeric(meta_df["train_wall_clock_seconds"], errors="coerce")
    run_times = pd.to_numeric(meta_df["run_wall_clock_seconds"], errors="coerce")

    timing_rows.append({
        "Model": name,
        "Best epoch mean": round(float(best_epochs.mean()), 1),
        "Best epoch std": round(float(best_epochs.std(ddof=1)), 1),
        "Epochs run mean": round(float(epochs_ran.mean()), 1),
        "Epochs run std": round(float(epochs_ran.std(ddof=1)), 1),
        "Training wall-clock time per seed (s) mean": round(float(train_times.mean()), 2),
        "Training wall-clock time per seed (s) std": round(float(train_times.std(ddof=1)), 2),
        "Total training wall-clock time, 5 seeds (s)": round(float(train_times.sum()), 2),
        "Full run wall-clock time per seed (s) mean": round(float(run_times.mean()), 2),
        "Full run wall-clock time per seed (s) std": round(float(run_times.std(ddof=1)), 2),
        "Total full run wall-clock time, 5 seeds (s)": round(float(run_times.sum()), 2),
    })

df_training_metadata = pd.concat(metadata_frames, ignore_index=True)
df_training_wallclock = pd.DataFrame(timing_rows)

df_training_metadata.to_csv("training_metadata_5seeds.csv", index=False)
df_training_wallclock.to_csv("training_wallclock_summary_5seeds.csv", index=False)

print("\n=== Real Training Metadata by Seed ===")
display(df_training_metadata)
print("\n=== Real Wall-Clock Training Cost Summary ===")
display(df_training_wallclock)
print("Saved: training_metadata_5seeds.csv")
print("Saved: training_wallclock_summary_5seeds.csv")


In [ ]:
# Inference latency summary

import time

N_INF_WARMUP = 10
N_INF_RUNS   = 50
BATCH_INF    = 1   # single-sample latency (deployment relevant)

@torch.no_grad()
def measure_inference_latency(model, ema_obj, batch_size=BATCH_INF,
                               n_warmup=N_INF_WARMUP, n_runs=N_INF_RUNS):
    """
    Measure inference latency per sample (ms).
    Use EMA weights as in deployment.
    """
    backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
    ema_obj.load_shadow(model)
    model.eval()

    x_sample = X_main_test_tensor[:batch_size].to(device)
    m_sample  = month_main_test_tensor[:batch_size].to(device)

    for _ in range(n_warmup):
        _ = model(x_sample, month_idx=m_sample)

    if device.type == "cuda":
        torch.cuda.synchronize()

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = model(x_sample, month_idx=m_sample)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000.0)  # ms

    model.load_state_dict(backup, strict=True)
    return float(np.mean(times)), float(np.std(times))

inf_rows = []
for name, (art_dict, seed) in [
    ("Pure CNN-BiLSTM-Attn",           (pure_artifacts, SEEDS[0])),
    ("Hybrid Anchor-Residual",         (hyb_artifacts,  SEEDS[0])),
    ("MoE-Gated Hybrid",               (moe_artifacts,  SEEDS[0])),
    ("Latent Physical Regularization", (lat_artifacts,  SEEDS[0])),
    ("WeatherCalib Pre-Net",           (wc_artifacts,   SEEDS[0])),
]:
    mdl      = art_dict[seed]["model"]
    ema_obj  = art_dict[seed]["ema"]
    mean_ms, std_ms = measure_inference_latency(mdl, ema_obj)
    print(f"{name}: {mean_ms:.3f} ± {std_ms:.3f} ms/sample")
    inf_rows.append({
        "Model":                     name,
        "Inference latency (ms) mean": round(mean_ms, 3),
        "Inference latency (ms) std":  round(std_ms,  3),
    })

df_inference = pd.DataFrame(inf_rows)
print("\n=== Inference Latency (single sample) ===")
display(df_inference)


In [ ]:
# Computational-cost summary table

df_cost_summary = (
    df_params
    .merge(df_training_wallclock, on="Model")
    .merge(df_inference, on="Model")
)

inference_path_note = {
    "Pure CNN-BiLSTM-Attn":           "Standard",
    "Hybrid Anchor-Residual":         "+Physics module",
    "MoE-Gated Hybrid":               "+Physics+Gate",
    "Latent Physical Regularization": "Standard (aux head training-only)",
    "WeatherCalib Pre-Net":           "+Calib (small pre-net)",
}
df_cost_summary["Inference path"] = df_cost_summary["Model"].map(inference_path_note)

paper_cols = [
    "Model",
    "Total params",
    "Trainable params",
    "Best epoch mean",
    "Epochs run mean",
    "Training wall-clock time per seed (s) mean",
    "Total training wall-clock time, 5 seeds (s)",
    "Inference latency (ms) mean",
    "Inference path",
]
df_cost_paper = df_cost_summary[paper_cols].copy()

print("\n=== TABLE II — Real Computational Cost Summary ===")
display(df_cost_summary)
df_cost_summary.to_csv("table_ii_computational_cost.csv", index=False)
df_cost_paper.to_csv("table_ii_computational_cost_paper.csv", index=False)
print("Saved: table_ii_computational_cost.csv")
print("Saved: table_ii_computational_cost_paper.csv")


In [ ]:
# NWP-source sensitivity: load Open-Meteo forecast file and build PRED_TEST tensor

import os
import numpy as np
import pandas as pd
import torch

OPENMETEO_SOURCE_NAME = "Open-Meteo historical forecast"

# For the repository version, the alternative Open-Meteo file is expected to be provided
# as a ready-made CSV rather than downloaded during review. The preferred filename is
# sample_file3.csv. The alternative sample_test3.csv name is also accepted for compatibility.
ALT_NWP_CANDIDATES = [
    "sample_file3.csv",
    "sample_test3.csv",
    "data_sample/sample_file3.csv",
    "data_sample/sample_test3.csv",
    "openmeteo_alternative_nwp_daily.csv",
]

def find_alt_nwp_file(candidates=ALT_NWP_CANDIDATES):
    """Return the first available alternative NWP CSV file."""
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        "No alternative Open-Meteo NWP file was found. Expected one of: "
        + ", ".join(candidates)
    )

def weather_code_to_condition(code):
    """Map Open-Meteo WMO weather codes to compact Visual Crossing-like conditions."""
    if pd.isna(code):
        return "Unknown"

    code = int(code)

    if code == 0:
        return "Clear"
    if code in [1, 2]:
        return "Partially cloudy"
    if code == 3:
        return "Overcast"
    if code in [45, 48]:
        return "Fog"
    if code in [51, 53, 55, 56, 57, 61, 63, 65, 66, 67, 80, 81, 82]:
        return "Rain"
    if code in [71, 73, 75, 77, 85, 86]:
        return "Snow"
    if code in [95, 96, 99]:
        return "Thunderstorm"

    return "Other"

def weather_code_to_description(code):
    """Map Open-Meteo WMO weather codes to short text descriptions."""
    if pd.isna(code):
        return "Unknown weather conditions"

    code = int(code)

    description_map = {
        0: "Clear sky",
        1: "Mainly clear conditions",
        2: "Partly cloudy conditions",
        3: "Overcast conditions",
        45: "Foggy conditions",
        48: "Depositing rime fog",
        51: "Light drizzle",
        53: "Moderate drizzle",
        55: "Dense drizzle",
        56: "Light freezing drizzle",
        57: "Dense freezing drizzle",
        61: "Slight rain",
        63: "Moderate rain",
        65: "Heavy rain",
        66: "Light freezing rain",
        67: "Heavy freezing rain",
        71: "Slight snow fall",
        73: "Moderate snow fall",
        75: "Heavy snow fall",
        77: "Snow grains",
        80: "Slight rain showers",
        81: "Moderate rain showers",
        82: "Violent rain showers",
        85: "Slight snow showers",
        86: "Heavy snow showers",
        95: "Thunderstorm",
        96: "Thunderstorm with slight hail",
        99: "Thunderstorm with heavy hail",
    }

    return description_map.get(code, "Other weather conditions")

def infer_preciptype(row):
    """Infer a Visual Crossing-like preciptype field from precipitation, temperature, and WMO code."""
    precip = row.get("precip", np.nan)
    temp = row.get("temp", np.nan)
    code = row.get("weather_code", np.nan)

    if pd.isna(precip) or precip <= 0:
        return ""

    if not pd.isna(code):
        code = int(code)

        if code in [71, 73, 75, 77, 85, 86]:
            return "snow"
        if code in [56, 57, 66, 67]:
            return "freezingrain"
        if code in [51, 53, 55, 61, 63, 65, 80, 81, 82, 95, 96, 99]:
            return "rain"

    if not pd.isna(temp) and temp <= 0:
        return "snow"

    return "rain"

def standardize_openmeteo_schema(df_raw):
    """
    Standardize the ready-made Open-Meteo CSV to the Visual Crossing-like schema used
    by the notebook.

    The file may already contain the final columns. If it contains Open-Meteo native
    columns, the required project columns are derived when possible. Missing fields that
    are not used by the model are retained as NaN/empty values for schema compatibility.
    """
    df = df_raw.copy()

    if "datetime" not in df.columns:
        if "time" in df.columns:
            df["datetime"] = df["time"]
        else:
            raise ValueError("The Open-Meteo file must contain either 'datetime' or 'time'.")

    df["datetime"] = pd.to_datetime(df["datetime"]).dt.normalize()

    # Convert Open-Meteo native names if they are present.
    rename_map = {
        "temperature_2m_max": "tempmax",
        "temperature_2m_min": "tempmin",
        "temperature_2m_mean": "temp",
        "dew_point_2m_mean": "dew",
        "relative_humidity_2m_mean": "humidity",
        "precipitation_sum": "precip",
        "precipitation_probability_max": "precipprob",
        "precipitation_hours": "precipitation_hours",
        "cloud_cover_mean": "cloudcover",
        "wind_gusts_10m_max": "windgust",
        "wind_speed_10m_mean": "windspeed",
        "wind_direction_10m_dominant": "winddir",
        "pressure_msl_mean": "sealevelpressure",
        "shortwave_radiation_sum": "shortwave_radiation_sum",
        "daylight_duration": "daylight_duration",
    }
    for old, new in rename_map.items():
        if old in df.columns and new not in df.columns:
            df[new] = df[old]

    # Derive solar variables from Open-Meteo shortwave radiation if needed.
    if "solarenergy" not in df.columns and "shortwave_radiation_sum" in df.columns:
        # Open-Meteo shortwave_radiation_sum is MJ/m²/day.
        df["solarenergy"] = pd.to_numeric(df["shortwave_radiation_sum"], errors="coerce") * 0.277778

    if "solarradiation" not in df.columns and "shortwave_radiation_sum" in df.columns:
        # Daily average W/m²: MJ/m²/day * 1,000,000 / 86400 = 11.5741 W/m².
        df["solarradiation"] = pd.to_numeric(df["shortwave_radiation_sum"], errors="coerce") * 11.5741

    if "precipcover" not in df.columns and "precipitation_hours" in df.columns:
        df["precipcover"] = (
            pd.to_numeric(df["precipitation_hours"], errors="coerce") / 24.0 * 100.0
        ).clip(0, 100)

    if "Day_Length" not in df.columns and "daylight_duration" in df.columns:
        df["Day_Length"] = pd.to_numeric(df["daylight_duration"], errors="coerce") / 3600.0

    # Map weather code to categorical text fields if needed.
    if "conditions" not in df.columns and "weather_code" in df.columns:
        df["conditions"] = pd.to_numeric(df["weather_code"], errors="coerce").apply(weather_code_to_condition)

    if "description" not in df.columns and "weather_code" in df.columns:
        df["description"] = pd.to_numeric(df["weather_code"], errors="coerce").apply(weather_code_to_description)

    if "preciptype" not in df.columns:
        if "weather_code" not in df.columns:
            df["weather_code"] = np.nan
        df["preciptype"] = df.apply(infer_preciptype, axis=1)

    # Add compatibility columns if not present. Only the columns included in base_features
    # affect the model. Other columns document the Visual Crossing-like schema.
    final_cols = [
        "datetime",
        "tempmax",
        "tempmin",
        "temp",
        "dew",
        "humidity",
        "precip",
        "precipprob",
        "precipcover",
        "preciptype",
        "windgust",
        "windspeed",
        "winddir",
        "sealevelpressure",
        "cloudcover",
        "visibility",
        "solarradiation",
        "solarenergy",
        "uvindex",
        "sunrise",
        "sunset",
        "conditions",
        "description",
        "Day_Length",
    ]

    for col in final_cols:
        if col not in df.columns:
            df[col] = "" if col in ["preciptype", "sunrise", "sunset", "conditions", "description"] else np.nan

    # Numeric conversion for numeric-like fields
    numeric_cols = [
        "tempmax", "tempmin", "temp", "dew", "humidity", "precip", "precipprob",
        "precipcover", "windgust", "windspeed", "winddir", "sealevelpressure",
        "cloudcover", "visibility", "solarradiation", "solarenergy", "uvindex",
        "Day_Length",
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df[final_cols].sort_values("datetime").drop_duplicates("datetime", keep="last").reset_index(drop=True)
    return df

def add_engineered_features_for_existing_schema(df_in, base_features):
    """Recompute engineered features and align the frame to the feature schema fitted in the main pipeline."""
    df_feat = df_in.copy().sort_values("datetime").reset_index(drop=True)
    df_feat["datetime"] = pd.to_datetime(df_feat["datetime"])

    numeric_cols = [
        "Generated",
        "solarenergy",
        "solarradiation",
        "tempmax",
        "tempmin",
        "temp",
        "dew",
        "humidity",
        "cloudcover",
        "precip",
        "precipprob",
        "precipcover",
        "windgust",
        "windspeed",
        "winddir",
        "sealevelpressure",
        "visibility",
        "uvindex",
        "Day_Length",
    ]

    for col in numeric_cols:
        if col in df_feat.columns:
            df_feat[col] = pd.to_numeric(df_feat[col], errors="coerce")

    doy = df_feat["datetime"].dt.dayofyear
    df_feat["DOY_sin"] = np.sin(2 * np.pi * doy / 365.0)
    df_feat["DOY_cos"] = np.cos(2 * np.pi * doy / 365.0)

    df_feat["month"] = df_feat["datetime"].dt.month
    df_feat["Month_sin"] = np.sin(2 * np.pi * df_feat["month"] / 12.0)
    df_feat["Month_cos"] = np.cos(2 * np.pi * df_feat["month"] / 12.0)

    if "humidity" in df_feat.columns and "windspeed" in df_feat.columns:
        df_feat["humidity_wind"] = df_feat["humidity"] * df_feat["windspeed"]

    # Align categorical weather conditions to the original one-hot encoded schema.
    if "conditions" in df_feat.columns:
        cond = df_feat["conditions"].fillna("Unknown").astype(str).str.strip()
        cond_dummies = pd.get_dummies(cond, prefix="cond", dtype=np.float32)
        df_feat = pd.concat([df_feat, cond_dummies], axis=1)

    df_feat["Generated_shift1"] = df_feat["Generated"].shift(1)

    for col in ["tempmax", "humidity", "solarenergy", "Generated"]:
        if col in df_feat.columns:
            df_feat[f"{col}_ewm3"] = df_feat[col].shift(1).ewm(span=3, adjust=False).mean()
            df_feat[f"{col}_ewm7"] = df_feat[col].shift(1).ewm(span=7, adjust=False).mean()

    # Physics proxy features
    A_total_m2 = 65.0
    eta_ref    = 0.15
    NOCT       = 45.0
    gamma_p    = -0.004
    c_wind     = 0.7
    PR_base    = 0.82

    df_feat["G_avg"] = np.clip(df_feat["solarenergy"] / 24.0, 0.0, 1.2)

    Tamb = df_feat["tempmax"].astype(float)
    wind = df_feat["windspeed"].fillna(0).astype(float)

    df_feat["T_cell_est"] = Tamb + (NOCT - 20.0) * df_feat["G_avg"] - c_wind * wind
    df_feat["f_T_est"] = np.clip(1.0 + gamma_p * (df_feat["T_cell_est"] - 25.0), 0.70, 1.15)
    df_feat["E_dc_est"] = np.clip(df_feat["solarenergy"] * A_total_m2 * eta_ref * df_feat["f_T_est"], 0.0, None)
    df_feat["PR_est_base"] = PR_base

    # Add any missing feature columns from the original schema as zeros.
    for col in base_features:
        if col not in df_feat.columns:
            df_feat[col] = 0.0

    needed = ["Generated", "datetime"] + list(base_features)
    df_feat = df_feat.dropna(subset=needed).reset_index(drop=True)

    return df_feat

def build_pred_tensor_from_alternative_weather(
    alt_weather_df,
    original_filtered_path="Dataset_PINN_Input_filtered.csv"
):
    """
    Replace PRED_TEST weather variables with an alternative forecast source, rebuild sequences,
    and transform them using the original training-only scalers. The scaler is not refitted.
    """

    base_df = pd.read_csv(original_filtered_path, parse_dates=["datetime"]).sort_values("datetime").reset_index(drop=True)
    base_df["datetime"] = pd.to_datetime(base_df["datetime"]).dt.normalize()

    alt = alt_weather_df.copy()
    alt["datetime"] = pd.to_datetime(alt["datetime"]).dt.normalize()
    alt = alt.drop_duplicates("datetime", keep="last")

    replace_cols = [
        "tempmax",
        "tempmin",
        "temp",
        "dew",
        "humidity",
        "precip",
        "precipprob",
        "precipcover",
        "preciptype",
        "windgust",
        "windspeed",
        "winddir",
        "sealevelpressure",
        "cloudcover",
        "visibility",
        "solarradiation",
        "solarenergy",
        "uvindex",
        "sunrise",
        "sunset",
        "Day_Length",
        "conditions",
        "description",
    ]

    alt = alt[[c for c in ["datetime"] + replace_cols if c in alt.columns]].copy()

    merged_alt = base_df.merge(alt, on="datetime", how="left", suffixes=("", "_alt"))
    pred_mask = merged_alt["datetime"] >= PRED_START

    for col in replace_cols:
        alt_col = f"{col}_alt"
        if alt_col in merged_alt.columns:
            merged_alt.loc[pred_mask & merged_alt[alt_col].notna(), col] = merged_alt.loc[
                pred_mask & merged_alt[alt_col].notna(), alt_col
            ]
            merged_alt = merged_alt.drop(columns=[alt_col])

    df_feat_alt = add_engineered_features_for_existing_schema(merged_alt, base_features)

    X_seq_alt, y_seq_alt, dates_alt = [], [], []

    for i in range(sequence_length_past, len(df_feat_alt)):
        start = i - sequence_length_past
        end = i + 1 if include_target_in_seq else i

        x_window = df_feat_alt.iloc[start:end][base_features].values.astype(np.float32)

        if include_target_in_seq and x_window.shape[0] != S:
            continue

        X_seq_alt.append(x_window)
        y_seq_alt.append(float(df_feat_alt.iloc[i][target]))
        dates_alt.append(df_feat_alt.iloc[i]["datetime"])

    X_seq_alt = np.asarray(X_seq_alt, dtype=np.float32)
    y_seq_alt = np.asarray(y_seq_alt, dtype=np.float32).reshape(-1, 1)
    dates_alt = pd.to_datetime(dates_alt)

    pred_mask_alt = dates_alt >= PRED_START

    X_pred_alt_raw = X_seq_alt[pred_mask_alt]
    y_pred_alt_raw = y_seq_alt[pred_mask_alt]
    dates_pred_alt = dates_alt[pred_mask_alt]

    X_pred_alt = scaler_X.transform(
        X_pred_alt_raw.reshape(-1, X_pred_alt_raw.shape[2])
    ).reshape(X_pred_alt_raw.shape)

    y_pred_alt = scaler_y.transform(np.log1p(y_pred_alt_raw))

    X_pred_alt_tensor = torch.tensor(X_pred_alt, dtype=torch.float32)
    y_pred_alt_tensor = torch.tensor(y_pred_alt, dtype=torch.float32)
    month_pred_alt_tensor = torch.tensor(dates_pred_alt.month.astype(np.int64).to_numpy(), dtype=torch.long)

    return X_pred_alt_tensor, y_pred_alt_tensor, month_pred_alt_tensor, dates_pred_alt

# Load ready-made Open-Meteo forecast file and build the PRED_TEST tensor.
OPENMETEO_AVAILABLE = False

try:
    alt_nwp_path = find_alt_nwp_file()
    print(f"Loading alternative NWP forecast file: {alt_nwp_path}")

    df_om_raw = pd.read_csv(alt_nwp_path)
    df_om = standardize_openmeteo_schema(df_om_raw)

    # Save standardized files for reproducibility outputs.
    df_om.to_csv("openmeteo_alternative_nwp_daily.csv", index=False)
    df_om.to_csv("sample_file3.csv", index=False)
    os.makedirs("data_sample", exist_ok=True)
    df_om.to_csv("data_sample/sample_file3.csv", index=False)

    X_pred_openmeteo_tensor, y_pred_openmeteo_tensor, month_pred_openmeteo_tensor, dates_pred_openmeteo = \
        build_pred_tensor_from_alternative_weather(df_om)

    print("Open-Meteo alternative NWP data prepared:")
    print(" - openmeteo_alternative_nwp_daily.csv")
    print(" - sample_file3.csv")
    print(" - data_sample/sample_file3.csv")
    print(f"Open-Meteo PRED_TEST tensor shape: {tuple(X_pred_openmeteo_tensor.shape)}")
    print(f"Open-Meteo date range: {dates_pred_openmeteo.min()} -> {dates_pred_openmeteo.max()}")

    OPENMETEO_AVAILABLE = True

except Exception as exc:
    print("Open-Meteo alternative NWP preparation failed.")
    print("Reason:", repr(exc))
    print("The notebook will continue with the original Visual Crossing PRED_TEST data only.")


In [ ]:
# Evaluate Visual Crossing original vs Open-Meteo alternative forecast on PRED_TEST

rep_models = {
    "Pure CNN-BiLSTM-Attn":           (pure_artifacts,  pure_runs),
    "Hybrid Anchor-Residual":         (hyb_artifacts,   hyb_runs),
    "MoE-Gated Hybrid":               (moe_artifacts,   moe_runs),
    "Latent Physical Regularization": (lat_artifacts,   lat_runs),
    "WeatherCalib Pre-Net":           (wc_artifacts,    wc_runs),
}

source_eval_inputs = {
    "Visual Crossing original": {
        "X": X_pred_test_tensor,
        "y": y_pred_test_true_tensor,
        "month": month_pred_test_tensor,
    }
}

if OPENMETEO_AVAILABLE:
    source_eval_inputs[OPENMETEO_SOURCE_NAME] = {
        "X": X_pred_openmeteo_tensor,
        "y": y_pred_openmeteo_tensor,
        "month": month_pred_openmeteo_tensor,
    }

nwp_rows = []

for model_name, (art_dict, run_dicts) in rep_models.items():
    rep_seed, _ = pick_representative_seed(run_dicts, key="score")
    mdl = art_dict[rep_seed]["model"]
    ema_obj = art_dict[rep_seed]["ema"]

    for source_name, inputs in source_eval_inputs.items():
        y_pred_source = model_predict_real(
            mdl,
            inputs["X"],
            month_idx=inputs["month"],
            use_ema=True,
            ema=ema_obj,
        )

        y_true_source = scaled_to_real_torch(
            inputs["y"], mdl, to_cpu=True
        ).squeeze(-1).detach()

        metrics = compute_metrics(y_true_source, y_pred_source)

        nwp_rows.append({
            "Model": model_name,
            "NWP Source": source_name,
            "RMSE_pred": round(metrics["RMSE"], 3),
            "MAE_pred": round(metrics["MAE"], 3),
            "R2_pred": round(metrics["R2"], 3),
        })

        print(f"{model_name} | {source_name} -> RMSE={metrics['RMSE']:.3f}")

df_nwp = pd.DataFrame(nwp_rows)

df_vc = (
    df_nwp[df_nwp["NWP Source"] == "Visual Crossing original"][["Model", "RMSE_pred"]]
    .rename(columns={"RMSE_pred": "RMSE_VC"})
)

df_nwp = df_nwp.merge(df_vc, on="Model", how="left")
df_nwp["Delta_RMSE_vs_VC"] = (df_nwp["RMSE_pred"] - df_nwp["RMSE_VC"]).round(3)

df_nwp = df_nwp.sort_values(["Model", "NWP Source"]).reset_index(drop=True)

print("\n=== NWP source sensitivity on PRED_TEST ===")
display(df_nwp)

# Main file requested for the paper/repository.
df_nwp.to_csv("table_nwp_source_sensitivity.csv", index=False)

# Backward-compatible name if earlier manuscript/table numbering used Table IV.
df_nwp.to_csv("table_iv_nwp_source_sensitivity.csv", index=False)

os.makedirs("outputs", exist_ok=True)
df_nwp.to_csv("outputs/table_nwp_source_sensitivity.csv", index=False)

print("Saved:")
print(" - table_nwp_source_sensitivity.csv")
print(" - table_iv_nwp_source_sensitivity.csv")
print(" - outputs/table_nwp_source_sensitivity.csv")


In [ ]:
# Plot NWP-source sensitivity

import os
import numpy as np
import matplotlib.pyplot as plt

if "df_nwp" not in globals() or df_nwp.empty:
    raise RuntimeError("df_nwp is not available. Run the NWP-source evaluation cell first.")

model_names = [
    "Pure CNN-BiLSTM-Attn",
    "Hybrid Anchor-Residual",
    "MoE-Gated Hybrid",
    "Latent Physical Regularization",
    "WeatherCalib Pre-Net",
]

available_sources = [s for s in ["Visual Crossing original", OPENMETEO_SOURCE_NAME] if s in df_nwp["NWP Source"].unique()]
x = np.arange(len(model_names))
width = 0.36 if len(available_sources) == 2 else 0.50

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=False)

# Panel A: absolute RMSE by NWP source
for i, source_name in enumerate(available_sources):
    vals = []
    for model_name in model_names:
        row = df_nwp[(df_nwp["Model"] == model_name) & (df_nwp["NWP Source"] == source_name)]
        vals.append(float(row["RMSE_pred"].iloc[0]) if len(row) else np.nan)

    offset = (i - (len(available_sources) - 1) / 2) * width
    axes[0].bar(x + offset, vals, width, label=source_name, alpha=0.90)

axes[0].set_xticks(x)
axes[0].set_xticklabels([m.replace(" ", "\n") for m in model_names], fontsize=8)
axes[0].set_ylabel("RMSE (kWh/day)")
axes[0].set_title("PRED_TEST RMSE by NWP source")
axes[0].grid(axis="y", alpha=0.3)
axes[0].legend(fontsize=8)

# Panel B: extra degradation relative to Visual Crossing
for i, source_name in enumerate(available_sources):
    vals = []
    for model_name in model_names:
        row = df_nwp[(df_nwp["Model"] == model_name) & (df_nwp["NWP Source"] == source_name)]
        vals.append(float(row["Delta_RMSE_vs_VC"].iloc[0]) if len(row) else np.nan)

    offset = (i - (len(available_sources) - 1) / 2) * width
    axes[1].bar(x + offset, vals, width, label=source_name, alpha=0.90)

axes[1].set_xticks(x)
axes[1].set_xticklabels([m.replace(" ", "\n") for m in model_names], fontsize=8)
axes[1].set_ylabel("Delta RMSE (kWh/day)")
axes[1].set_title("Extra degradation vs Visual Crossing")
axes[1].grid(axis="y", alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()

plt.savefig("fig_nwp_source_sensitivity.pdf", bbox_inches="tight")
plt.savefig("fig_nwp_source_sensitivity.png", dpi=300, bbox_inches="tight")

os.makedirs("outputs", exist_ok=True)
plt.savefig("outputs/fig_nwp_source_sensitivity.pdf", bbox_inches="tight")
plt.savefig("outputs/fig_nwp_source_sensitivity.png", dpi=300, bbox_inches="tight")

plt.show()

print("Saved:")
print(" - fig_nwp_source_sensitivity.pdf")
print(" - fig_nwp_source_sensitivity.png")
print(" - outputs/fig_nwp_source_sensitivity.pdf")
print(" - outputs/fig_nwp_source_sensitivity.png")


In [ ]:
# Season labels for main and predicted-weather test sets

import pandas as pd
import numpy as np
import torch

def assign_season(month_series, mode="standard"):
    """
    mode='standard': Dec-Feb=Winter, Jun-Aug=Summer, Mar-May=Spring, Sep-Nov=Autumn
    mode='pred':     Sep-Nov=Autumn, Dec-Feb=Winter (PRED_TEST only)
    """
    m = month_series
    if mode == "standard":
        return pd.cut(
            m,
            bins=[0, 2, 5, 8, 11, 12],
            labels=["Winter","Spring","Summer","Autumn","Winter2"],
            ordered=False
        ).astype(str).replace("Winter2", "Winter")
    elif mode == "pred":
        conditions = [
            m.between(9, 11),   # Autumn
            m.isin([12, 1, 2]), # Winter
        ]
        choices = ["Autumn (Sep-Nov)", "Winter (Dec-Feb)"]
        return np.select(conditions, choices, default="Other")
    return m.astype(str)

def build_prediction_df(split_name, dates_split, X_split, y_split, month_split):
    """
    Build a DataFrame with predictions from all models and true values
    for one segment (MAIN_TEST or PRED_TEST).
    """
    y_true = scaled_to_real_torch(
        y_split, pure_artifacts[SEEDS[0]]["model"], to_cpu=True
    ).squeeze(-1).detach().numpy()

    df = pd.DataFrame({
        "date":  pd.to_datetime(dates_split),
        "split": split_name,
        "month": pd.to_datetime(dates_split).month,
        "actual": y_true,
    })

    model_map = {
        "Pure CNN-BiLSTM-Attn":           (pure_artifacts, pure_runs),
        "Hybrid Anchor-Residual":         (hyb_artifacts,  hyb_runs),
        "MoE-Gated Hybrid":               (moe_artifacts,  moe_runs),
        "Latent Physical Regularization": (lat_artifacts,  lat_runs),
        "WeatherCalib Pre-Net":           (wc_artifacts,   wc_runs),
    }

    for mname, (art_dict, run_dicts) in model_map.items():
        rep_seed, _ = pick_representative_seed(run_dicts, key="score")
        mdl     = art_dict[rep_seed]["model"]
        ema_obj = art_dict[rep_seed]["ema"]
        y_hat   = model_predict_real(
            mdl, X_split, month_idx=month_split,
            use_ema=True, ema=ema_obj
        ).detach().numpy()
        df[mname] = y_hat

    return df

df_main_seas = build_prediction_df(
    "MAIN_TEST", dates_main_test,
    X_main_test_tensor, y_main_test_tensor, month_main_test_tensor
)
df_pred_seas = build_prediction_df(
    "PRED_TEST", dates_pred_test,
    X_pred_test_tensor, y_pred_test_true_tensor, month_pred_test_tensor
)

df_main_seas["season"] = assign_season(df_main_seas["month"], mode="standard")
df_pred_seas["season"] = assign_season(df_pred_seas["month"], mode="pred")

print("MAIN_TEST seasons:", df_main_seas["season"].value_counts().to_dict())
print("PRED_TEST seasons:", df_pred_seas["season"].value_counts().to_dict())


In [ ]:
# Seasonal RMSE table

MODEL_COLS = [
    "Pure CNN-BiLSTM-Attn",
    "Hybrid Anchor-Residual",
    "MoE-Gated Hybrid",
    "Latent Physical Regularization",
    "WeatherCalib Pre-Net",
]

def seasonal_rmse_table(df, model_cols, split_label):
    rows = []
    for season, grp in df.groupby("season", observed=True):
        n = len(grp)
        row = {"Split": split_label, "Season": season, "N": n}
        for col in model_cols:
            err = grp["actual"].values - grp[col].values
            rmse = float(np.sqrt(np.mean(err**2)))
            row[col] = round(rmse, 3)
        rows.append(row)
    return pd.DataFrame(rows)

df_seas_main = seasonal_rmse_table(df_main_seas, MODEL_COLS, "MAIN_TEST")
df_seas_pred = seasonal_rmse_table(df_pred_seas, MODEL_COLS, "PRED_TEST")
df_seas_all  = pd.concat([df_seas_main, df_seas_pred], ignore_index=True)

print("\n=== TABLE III — Seasonal RMSE Breakdown ===")
display(df_seas_all)
df_seas_all.to_csv("table_iii_seasonal_rmse.csv", index=False)
print("Saved: table_iii_seasonal_rmse.csv")


In [ ]:
# Seasonal RMSE drift

def cross_split_delta_rmse(df_main, df_pred, model_cols):
    """
    Compare degradation (ΔRMSE) for each model within the season.
    MAIN_TEST: Winter = Dec-Feb
    PRED_TEST: Winter (Dec-Feb)
    """
    rows = []
    main_winter = df_main[df_main["season"].str.lower().str.contains("winter")]
    pred_winter = df_pred[df_pred["season"].str.lower().str.contains("winter")]

    for col in model_cols:
        err_main = main_winter["actual"].values - main_winter[col].values
        err_pred = pred_winter["actual"].values - pred_winter[col].values
        rmse_main_w = float(np.sqrt(np.mean(err_main**2)))
        rmse_pred_w = float(np.sqrt(np.mean(err_pred**2)))
        rows.append({
            "Model":         col,
            "RMSE_main_winter": round(rmse_main_w, 3),
            "RMSE_pred_winter": round(rmse_pred_w, 3),
            "ΔRMSE_winter":     round(rmse_pred_w - rmse_main_w, 3),
            "N_main": len(main_winter),
            "N_pred": len(pred_winter),
        })

    return pd.DataFrame(rows)

df_delta_winter = cross_split_delta_rmse(df_main_seas, df_pred_seas, MODEL_COLS)
print("\n=== Winter-specific ΔRMSE (Main Winter → Pred Winter) ===")
display(df_delta_winter)
df_delta_winter.to_csv("table_iii_b_winter_delta_rmse.csv", index=False)
print("Saved: table_iii_b_winter_delta_rmse.csv")

def autumn_vs_overall(df_main, df_pred, model_cols):
    rows = []
    main_overall_rmse = {}
    for col in model_cols:
        err = df_main["actual"].values - df_main[col].values
        main_overall_rmse[col] = float(np.sqrt(np.mean(err**2)))

    pred_autumn = df_pred[df_pred["season"].str.lower().str.contains("autumn")]
    for col in model_cols:
        err_pred = pred_autumn["actual"].values - pred_autumn[col].values
        rmse_pred_a = float(np.sqrt(np.mean(err_pred**2)))
        rows.append({
            "Model":             col,
            "RMSE_main_overall": round(main_overall_rmse[col], 3),
            "RMSE_pred_autumn":  round(rmse_pred_a, 3),
            "ΔRMSE_autumn":      round(rmse_pred_a - main_overall_rmse[col], 3),
            "N_pred_autumn":     len(pred_autumn),
        })
    return pd.DataFrame(rows)

df_delta_autumn = autumn_vs_overall(df_main_seas, df_pred_seas, MODEL_COLS)
print("\n=== Autumn ΔRMSE (Pred Autumn vs Main overall) ===")
display(df_delta_autumn)


In [ ]:
# Seasonal visualization

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

def normalize_season_table(df):
    df = df.copy()

    if "season" not in df.columns:
        if df.index.name is not None and str(df.index.name).lower() == "season":
            df = df.reset_index()
        elif "Season" in df.columns:
            df = df.rename(columns={"Season": "season"})
        elif "SEASON" in df.columns:
            df = df.rename(columns={"SEASON": "season"})
        else:
            first_col = df.columns[0]
            possible = df[first_col].astype(str).str.contains(
                "Winter|Spring|Summer|Autumn|Sep|Nov|Dec|Feb",
                case=False,
                regex=True
            ).any()
            if possible:
                df = df.rename(columns={first_col: "season"})
            else:
                df = df.reset_index()
                if "season" not in df.columns:
                    df = df.rename(columns={df.columns[0]: "season"})

    df["season"] = df["season"].astype(str).str.strip()
    return df

df_seas_main_plot = normalize_season_table(df_seas_main)
df_seas_pred_plot = normalize_season_table(df_seas_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

for ax, (df_s, title, seasons_order) in zip(
    axes,
    [
        (df_seas_main_plot, "MAIN_TEST - Seasonal RMSE by Model",
         ["Winter", "Spring", "Summer", "Autumn"]),
        (df_seas_pred_plot, "PRED_TEST - Seasonal RMSE by Model",
         ["Autumn (Sep-Nov)", "Winter (Dec-Feb)"]),
    ]
):
    n_seasons = len(seasons_order)
    n_models = len(MODEL_COLS)
    x = np.arange(n_models)
    width = 0.8 / max(n_seasons, 1)

    palette = ["#2E75B6", "#70AD47", "#ED7D31", "#FFC000", "#7030A0"]

    for i, season in enumerate(seasons_order):
        row = df_s[df_s["season"] == season]
        if row.empty:
            continue

        vals = [
            float(row[col].iloc[0]) if col in row.columns and pd.notna(row[col].iloc[0]) else np.nan
            for col in MODEL_COLS
        ]

        ax.bar(
            x + i * width,
            vals,
            width,
            label=season,
            color=palette[i % len(palette)],
            alpha=0.85
        )

    ax.set_xticks(x + (n_seasons - 1) * width / 2)
    ax.set_xticklabels([m.replace(" ", "\n") for m in MODEL_COLS], fontsize=8)
    ax.set_ylabel("RMSE (kWh/day)")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Seasonal RMSE Breakdown (Representative Seed per Model)", fontsize=11)
plt.tight_layout()
plt.savefig("fig_seasonal_rmse_breakdown.pdf", bbox_inches="tight")
plt.show()
print("Saved: fig_seasonal_rmse_breakdown.pdf")

required_cols = ["RMSE_main_winter", "RMSE_pred_winter", "ΔRMSE_winter"]
missing = [c for c in required_cols if c not in df_delta_winter.columns]

if missing:
    raise KeyError(
        f"df_delta_winter is missing columns: {missing}. "
        f"Available columns: {list(df_delta_winter.columns)}"
    )

df_delta_plot = df_delta_winter.copy().reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 4))

x = np.arange(len(MODEL_COLS))
width = 0.35

bars1 = ax.bar(
    x - width / 2,
    df_delta_plot["RMSE_main_winter"],
    width,
    label="RMSE_main Winter",
    color="#2E75B6",
    alpha=0.85
)

bars2 = ax.bar(
    x + width / 2,
    df_delta_plot["RMSE_pred_winter"],
    width,
    label="RMSE_pred Winter",
    color="#ED7D31",
    alpha=0.85
)

for xi, drmse in zip(x, df_delta_plot["ΔRMSE_winter"]):
    y_top = max(
        df_delta_plot.loc[xi, "RMSE_main_winter"],
        df_delta_plot.loc[xi, "RMSE_pred_winter"]
    )

    ax.annotate(
        f"Δ={drmse:+.2f}",
        xy=(xi, y_top + 0.15),
        ha="center",
        fontsize=8,
        color="#C00000",
        fontweight="bold"
    )

ax.set_xticks(x)
ax.set_xticklabels([m.replace(" ", "\n") for m in MODEL_COLS], fontsize=8)
ax.set_ylabel("RMSE (kWh/day)")
ax.set_title("Winter ΔRMSE: MAIN_TEST Winter vs PRED_TEST Winter", fontsize=10)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("fig_winter_delta_rmse.pdf", bbox_inches="tight")
plt.show()
print("Saved: fig_winter_delta_rmse.pdf")


In [ ]:
# Winter contribution to total RMSE drift

print("\n=== Winter Fraction of Total ΔRMSE ===")
for col in MODEL_COLS:
    try:
        rmse_main_all = df_seas_main.set_index("Season").loc[
            :, col].dropna().values  # all seasons
        rdict = {
            "Pure CNN-BiLSTM-Attn":           pure_runs,
            "Hybrid Anchor-Residual":         hyb_runs,
            "MoE-Gated Hybrid":               moe_runs,
            "Latent Physical Regularization": lat_runs,
            "WeatherCalib Pre-Net":           wc_runs,
        }
        median_delta_total = np.median([r["d_rmse"] for r in rdict[col]])
        delta_winter = df_delta_winter[df_delta_winter["Model"] == col]["ΔRMSE_winter"].values[0]
        fraction = 100.0 * delta_winter / median_delta_total if median_delta_total > 0 else np.nan
        print(f"  {col:<42}: ΔRMSE_winter={delta_winter:+.3f}  |  total ΔRMSE≈{median_delta_total:.3f}  |  fraction≈{fraction:.1f}%")
    except Exception as e:
        print(f"  {col}: {e}")


In [ ]:
# Persistence summary

LAMBDA_SCORE = 1.0
ROLL_WIN = 15
ROLL_STEP = 1

pers_res = eval_persistence_main_pred(
    baseline if "baseline" in globals() else build_pure_model(),
    X_main_test_tensor, y_main_test_tensor,
    X_pred_test_tensor, y_pred_test_true_tensor,
    lambda_score=LAMBDA_SCORE,
    name="Persistence D-1"
)

roll_main_pers = rolling_eval_persistence(
    baseline if "baseline" in globals() else build_pure_model(),
    X_main_test_tensor, y_main_test_tensor,
    win=ROLL_WIN, step=ROLL_STEP, label="MAIN_TEST"
)

roll_pred_pers = rolling_eval_persistence(
    baseline if "baseline" in globals() else build_pure_model(),
    X_pred_test_tensor, y_pred_test_true_tensor,
    win=ROLL_WIN, step=ROLL_STEP, label="PRED_TEST"
)

pers_results = {
    "Model": "Persistence D-1",
    "RMSE_main": pers_res["metrics_main"]["RMSE"],
    "RMSE_main_std": 0.0,
    "MAE_main": pers_res["metrics_main"]["MAE"],
    "MAE_main_std": 0.0,
    "R2_main": pers_res["metrics_main"]["R2"],
    "R2_main_std": 0.0,
    "MAPE_main": pers_res["metrics_main"]["MAPE>1"],
    "MAPE_main_std": 0.0,

    "RMSE_pred": pers_res["metrics_pred"]["RMSE"],
    "RMSE_pred_std": 0.0,
    "MAE_pred": pers_res["metrics_pred"]["MAE"],
    "MAE_pred_std": 0.0,
    "R2_pred": pers_res["metrics_pred"]["R2"],
    "R2_pred_std": 0.0,
    "MAPE_pred": pers_res["metrics_pred"]["MAPE>1"],
    "MAPE_pred_std": 0.0,

    "ΔRMSE": pers_res["dRMSE"],
    "ΔRMSE_std": 0.0,
    "Score": pers_res["score"],
    "Score_std": 0.0,
}
display(pd.DataFrame([pers_results]))


In [ ]:
# Final multi-run comparison

rows = []

rows.append(result_row_from_summary(df_lat_summary))
rows.append(result_row_from_summary(df_wc_summary))
rows.append(result_row_from_summary(df_pure_summary))
rows.append(result_row_from_summary(df_moe_summary))
rows.append(result_row_from_summary(df_hyb_summary))
rows.append(result_row_from_summary(df_hyb_phys_summary))
rows.append(pers_results)

df_final = pd.DataFrame(rows)
df_final = df_final.sort_values(["Score", "RMSE_pred"], ascending=[True, True]).reset_index(drop=True)

display(df_final)

df_final.to_csv("model_comparison_all_5seeds_mean_std.csv", index=False)
print("Saved: model_comparison_all_5seeds_mean_std.csv")


In [ ]:
# LaTeX table export

def fmt_pm(mean, std, nd=2):
    return f"{mean:.{nd}f} $\\pm$ {std:.{nd}f}"

def fmt_num(x, nd=3):
    return f"{x:.{nd}f}"

df_tab = df_final.sort_values("Score", ascending=True).reset_index(drop=True).copy()

latex_rows = []
for _, r in df_tab.iterrows():
    row = (
        f"{r['Model']} & "
        f"{fmt_pm(r['RMSE_main'], r['RMSE_main_std'], 2)} & "
        f"{fmt_num(r['R2_main'], 3)} & "
        f"{fmt_pm(r['RMSE_pred'], r['RMSE_pred_std'], 2)} & "
        f"{fmt_num(r['R2_pred'], 3)} & "
        f"{fmt_pm(r['ΔRMSE'], r['ΔRMSE_std'], 2)} & "
        f"{fmt_pm(r['Score'], r['Score_std'], 2)} \\\\"
    )
    latex_rows.append(row)

latex_table = r"""
\begin{table}[!t]
\caption{Comparison of nominal accuracy and robustness under forecasted-weather drift across five random seeds (mean $\pm$ std).}
\label{tab:main_results}
\centering
\renewcommand{\arraystretch}{1.08}
\setlength{\tabcolsep}{5pt}
\begin{tabular}{lcccccc}
\hline
\textbf{Model} & \multicolumn{2}{c}{\textbf{MAIN\_TEST}} & \multicolumn{2}{c}{\textbf{PRED\_TEST}} & \textbf{$\Delta$RMSE} & \textbf{Score} \\
\cline{2-3} \cline{4-5}
 & \textbf{RMSE $\pm$ std} & \textbf{$R^2$} & \textbf{RMSE $\pm$ std} & \textbf{$R^2$} & \textbf{mean $\pm$ std} & \textbf{mean $\pm$ std} \\
\hline
""" + "\n".join(latex_rows) + r"""
\hline
\end{tabular}
\end{table}
"""

print(latex_table)


In [ ]:
# Time-series overlay for representative Latent Regularization runs

import matplotlib.pyplot as plt
import torch
import numpy as np

rep_seed_lat, rep_idx_lat = pick_representative_seed(lat_runs, key="score")
print("Representative seed for Latent Reg:", rep_seed_lat, "| run index:", rep_idx_lat)

lat_rep = lat_artifacts[rep_seed_lat]["model"]
ema_lat_rep = lat_artifacts[rep_seed_lat]["ema"]

BATCH_EVAL = 4096

@torch.no_grad()
def get_latent_real_pred(model_obj, ema_obj, X_seg, y_seg, month_seg, batch_size=BATCH_EVAL):
    y_true = scaled_to_real_torch(y_seg, model_obj, to_cpu=True).squeeze(-1).detach()
    y_pred = model_predict_real(
        model_obj,
        X_seg,
        month_idx=month_seg,
        batch_size=batch_size,
        use_ema=True,
        ema=ema_obj
    ).detach().cpu()
    return y_true.numpy(), y_pred.numpy()

y_main_real, y_main_pred = get_latent_real_pred(
    lat_rep, ema_lat_rep,
    X_main_test_tensor, y_main_test_tensor, month_main_test_tensor
)
m_main = compute_metrics(torch.tensor(y_main_real), torch.tensor(y_main_pred))

y_pred_real, y_pred_pred = get_latent_real_pred(
    lat_rep, ema_lat_rep,
    X_pred_test_tensor, y_pred_test_true_tensor, month_pred_test_tensor
)
m_pred = compute_metrics(torch.tensor(y_pred_real), torch.tensor(y_pred_pred))

fig, axes = plt.subplots(2, 1, figsize=(14.5, 8.0), sharey=True)

ax = axes[0]
ax.plot(dates_main_test, y_main_real, label="Real")
ax.plot(dates_main_test, y_main_pred, label=f"Pred (Latent Reg, seed={rep_seed_lat})")
ax.set_title(
    f"MAIN_TEST (measured weather) — N={len(y_main_real)} | "
    f"RMSE={m_main['RMSE']:.3f}, MAE={m_main['MAE']:.3f}, R²={m_main['R2']:.3f}"
)
ax.set_ylabel("Energy (kWh/day)")
ax.grid(True, alpha=0.25)
ax.legend()

ax = axes[1]
ax.plot(dates_pred_test, y_pred_real, label="Real")
ax.plot(dates_pred_test, y_pred_pred, label=f"Pred (Latent Reg, seed={rep_seed_lat})")
ax.set_title(
    f"PRED_TEST (forecasted weather / drift regime) — N={len(y_pred_real)} | "
    f"RMSE={m_pred['RMSE']:.3f}, MAE={m_pred['MAE']:.3f}, R²={m_pred['R2']:.3f}"
)
ax.set_xlabel("Date")
ax.set_ylabel("Energy (kWh/day)")
ax.grid(True, alpha=0.25)
ax.legend()

fig.tight_layout()
fig.savefig("latentreg_timeseries_main_pred_5seeds_rep.pdf", bbox_inches="tight")
fig.savefig("latentreg_timeseries_main_pred_5seeds_rep.png", dpi=300, bbox_inches="tight")

plt.show()


In [ ]:
# Figure: five-run mean and standard deviation

import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("model_comparison_all_5seeds_mean_std.csv")

keep_models = [
    "Latent Physical Regularization",
    "WeatherCalib Pre-Net",
    "PURE CNN-BiLSTM-attn",
    "MoE-Gated Hybrid",
    "Hybrid Anchor--Residual",
]
df = df[df["Model"].isin(keep_models)].copy()

name_map = {
    "Hybrid Anchor--Residual": "Anchor-Residual",
    "PURE CNN-BiLSTM-attn": "Pure CNN-BiLSTM-attn",
    "Latent Physical Regularization": "Latent Reg",
    "WeatherCalib Pre-Net": "WeatherCalib",
    "MoE-Gated Hybrid": "MoE-Gated",
}
df["Label"] = df["Model"].map(name_map).fillna(df["Model"])

fig, ax = plt.subplots(figsize=(7.4, 5.4))

for _, row in df.iterrows():
    x = row["RMSE_main"]
    y = row["RMSE_pred"]
    xerr = row["RMSE_main_std"]
    yerr = row["RMSE_pred_std"]

    ax.errorbar(
        x, y,
        xerr=xerr,
        yerr=yerr,
        fmt="o",
        capsize=3,
        elinewidth=1.1,
        markersize=6
    )

offsets = {
    "WeatherCalib": (6, -10),
    "Latent Reg": (6, 6),
    "Pure CNN-BiLSTM-attn": (6, 6),
    "MoE-Gated": (6, 6),
    "Anchor-Residual": (6, 6),
}

for _, row in df.iterrows():
    dx, dy = offsets.get(row["Label"], (6, 6))
    ax.annotate(
        row["Label"],
        (row["RMSE_main"], row["RMSE_pred"]),
        xytext=(dx, dy),
        textcoords="offset points",
        fontsize=9
    )

ax.set_xlabel("RMSE on MAIN_TEST (mean ± std over 5 runs)", fontsize=10)
ax.set_ylabel("RMSE on PRED_TEST (mean ± std over 5 runs)", fontsize=10)
ax.grid(True, alpha=0.25)
ax.tick_params(labelsize=9)

xmin = (df["RMSE_main"] - df["RMSE_main_std"]).min() - 0.15
xmax = (df["RMSE_main"] + df["RMSE_main_std"]).max() + 0.15
ymin = (df["RMSE_pred"] - df["RMSE_pred_std"]).min() - 0.15
ymax = (df["RMSE_pred"] + df["RMSE_pred_std"]).max() + 0.15
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

fig.tight_layout()
fig.savefig("tradeoff_scatter_5runs_mean_std.pdf", bbox_inches="tight")
fig.savefig("tradeoff_scatter_5runs_mean_std.png", dpi=300, bbox_inches="tight")

plt.show()


In [ ]:
# Figure: weather-drift robustness

import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("model_comparison_all_5seeds_mean_std.csv")

keep_models = [
    "Latent Physical Regularization",
    "WeatherCalib Pre-Net",
    "PURE CNN-BiLSTM-attn",
    "MoE-Gated Hybrid",
    "Hybrid Anchor--Residual",
]
df = df[df["Model"].isin(keep_models)].copy()

name_map = {
    "Hybrid Anchor--Residual": "Anchor-Residual",
    "PURE CNN-BiLSTM-attn": "Pure CNN-BiLSTM-attn",
    "Latent Physical Regularization": "Latent Reg",
    "WeatherCalib Pre-Net": "WeatherCalib",
    "MoE-Gated Hybrid": "MoE-Gated",
}
df["Label"] = df["Model"].map(name_map).fillna(df["Model"])

df = df.sort_values("ΔRMSE", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(7.4, 4.8))

ypos = range(len(df))

x = df["ΔRMSE"]
xerr = df["ΔRMSE_std"]

ax.errorbar(
    x,
    ypos,
    xerr=xerr,
    fmt="o",
    capsize=3,
    elinewidth=1.1,
    markersize=6
)

ax.axvline(0, linestyle="--", linewidth=1)

ax.set_yticks(list(ypos))
ax.set_yticklabels(df["Label"], fontsize=9)
ax.set_xlabel(r"$\Delta$RMSE = RMSE$_{PRED}$ - RMSE$_{MAIN}$ (mean $\pm$ std over 5 runs)", fontsize=10)
ax.set_ylabel("")
ax.grid(True, axis="x", alpha=0.25)
ax.tick_params(labelsize=9)

fig.tight_layout()
fig.savefig("delta_rmse_5runs_mean_std.pdf", bbox_inches="tight")
fig.savefig("delta_rmse_5runs_mean_std.png", dpi=300, bbox_inches="tight")

plt.show()


In [ ]:
# Figure: rolling RMSE curves

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

ROLL_WIN = 15
ROLL_STEP = 1
BATCH_EVAL = 4096

@torch.no_grad()
def get_real_and_pred(model_obj, ema_obj, X_seg, y_seg, month_seg, batch_size=BATCH_EVAL):
    y_true = scaled_to_real_torch(y_seg, model_obj, to_cpu=True).squeeze(-1).detach()
    y_pred = model_predict_real(
        model_obj,
        X_seg,
        month_idx=month_seg,
        batch_size=batch_size,
        use_ema=True,
        ema=ema_obj
    ).detach().cpu()
    return y_true, y_pred

def rolling_rmse_series(y_true, y_pred, dates, win=15, step=1):
    y_true = torch.as_tensor(y_true).detach().cpu()
    y_pred = torch.as_tensor(y_pred).detach().cpu()

    rmses = []
    centers = []

    N = len(y_true)
    for s in range(0, N - win + 1, step):
        e = s + win
        m = compute_metrics(y_true[s:e], y_pred[s:e])
        rmses.append(m["RMSE"])
        centers.append(pd.to_datetime(dates[s + win // 2]))

    return pd.DataFrame({
        "date": centers,
        "rolling_rmse": rmses
    })

rep_seed_pure, _ = pick_representative_seed(pure_runs, key="score")
rep_seed_lat,  _ = pick_representative_seed(lat_runs,  key="score")
rep_seed_hyb,  _ = pick_representative_seed(hyb_runs,  key="score")

print("Representative seeds:")
print("  Pure  :", rep_seed_pure)
print("  Latent:", rep_seed_lat)
print("  Hybrid:", rep_seed_hyb)

pure_rep = pure_artifacts[rep_seed_pure]
lat_rep  = lat_artifacts[rep_seed_lat]
hyb_rep  = hyb_artifacts[rep_seed_hyb]

models_to_plot = [
    ("Pure CNN-BiLSTM", pure_rep["model"], pure_rep["ema"]),
    ("Latent Reg", lat_rep["model"], lat_rep["ema"]),
    ("Anchor-Residual", hyb_rep["model"], hyb_rep["ema"]),
]

rolling_main = {}
rolling_pred = {}

for label, mdl, ema_obj in models_to_plot:
    y_true_main, y_hat_main = get_real_and_pred(
        mdl, ema_obj,
        X_main_test_tensor, y_main_test_tensor, month_main_test_tensor
    )
    rolling_main[label] = rolling_rmse_series(
        y_true_main, y_hat_main, dates_main_test, win=ROLL_WIN, step=ROLL_STEP
    )

    y_true_pred, y_hat_pred = get_real_and_pred(
        mdl, ema_obj,
        X_pred_test_tensor, y_pred_test_true_tensor, month_pred_test_tensor
    )
    rolling_pred[label] = rolling_rmse_series(
        y_true_pred, y_hat_pred, dates_pred_test, win=ROLL_WIN, step=ROLL_STEP
    )

fig, axes = plt.subplots(2, 1, figsize=(15, 8.5), sharex=False, sharey=False)

ax = axes[0]
for label, df_roll in rolling_main.items():
    ax.plot(df_roll["date"], df_roll["rolling_rmse"], label=label, linewidth=1.8)
ax.set_title(f"Rolling RMSE on MAIN_TEST (window={ROLL_WIN}, step={ROLL_STEP})")
ax.set_ylabel("RMSE (kWh/day)")
ax.grid(True, alpha=0.25)
ax.legend()

ax = axes[1]
for label, df_roll in rolling_pred.items():
    ax.plot(df_roll["date"], df_roll["rolling_rmse"], label=label, linewidth=1.8)
ax.set_title(f"Rolling RMSE on PRED_TEST (window={ROLL_WIN}, step={ROLL_STEP})")
ax.set_xlabel("Date")
ax.set_ylabel("RMSE (kWh/day)")
ax.grid(True, alpha=0.25)
ax.legend()

plt.tight_layout()
plt.show()

fig.savefig("rolling_rmse_curves_pure_latent_hybrid.pdf", bbox_inches="tight")
fig.savefig("rolling_rmse_curves_pure_latent_hybrid.png", dpi=300, bbox_inches="tight")


In [ ]:
# Figure: PRED_TEST model comparison

import matplotlib.pyplot as plt
import torch

BATCH_EVAL = 4096

@torch.no_grad()
def get_real_pred_numpy(model_obj, ema_obj, X_seg, y_seg, month_seg, batch_size=BATCH_EVAL):
    y_true = scaled_to_real_torch(y_seg, model_obj, to_cpu=True).squeeze(-1).detach().cpu()
    y_pred = model_predict_real(
        model_obj,
        X_seg,
        month_idx=month_seg,
        batch_size=batch_size,
        use_ema=True,
        ema=ema_obj
    ).detach().cpu()
    return y_true.numpy(), y_pred.numpy()

rep_seed_pure, _ = pick_representative_seed(pure_runs, key="score")
rep_seed_lat,  _ = pick_representative_seed(lat_runs,  key="score")
rep_seed_hyb,  _ = pick_representative_seed(hyb_runs,  key="score")

pure_rep = pure_artifacts[rep_seed_pure]
lat_rep  = lat_artifacts[rep_seed_lat]
hyb_rep  = hyb_artifacts[rep_seed_hyb]

models_to_plot = [
    ("Pure CNN-BiLSTM", pure_rep["model"], pure_rep["ema"], rep_seed_pure),
    ("Latent Reg", lat_rep["model"], lat_rep["ema"], rep_seed_lat),
    ("Anchor-Residual", hyb_rep["model"], hyb_rep["ema"], rep_seed_hyb),
]

y_true_np = scaled_to_real_torch(
    y_pred_test_true_tensor,
    pure_rep["model"],
    to_cpu=True
).squeeze(-1).detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(15, 6.5))

ax.plot(dates_pred_test, y_true_np, label="Real", linewidth=2.2)

metrics_lines = []

for label, mdl, ema_obj, seed_used in models_to_plot:
    _, y_hat_np = get_real_pred_numpy(
        mdl, ema_obj,
        X_pred_test_tensor, y_pred_test_true_tensor, month_pred_test_tensor
    )

    m = compute_metrics(torch.tensor(y_true_np), torch.tensor(y_hat_np))

    ax.plot(
        dates_pred_test,
        y_hat_np,
        linewidth=1.6,
        alpha=0.95,
        label=f"{label}"
    )

    metrics_lines.append(
        f"{label} (seed={seed_used}): "
        f"RMSE={m['RMSE']:.3f}, MAE={m['MAE']:.3f}, "
        f"R²={m['R2']:.3f}, MAPE>1={m['MAPE>1']:.2f}%"
    )

ax.set_title("PRED_TEST Comparison: Real vs Pure CNN-BiLSTM vs Latent Reg vs Anchor-Residual")
ax.set_xlabel("Date")
ax.set_ylabel("Energy (kWh/day)")
ax.grid(True, alpha=0.25)
ax.legend(loc="upper right", ncol=2)

metrics_text = "\n".join(metrics_lines)
ax.text(
    0.01, 0.99, metrics_text,
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
)

plt.tight_layout()
plt.show()

fig.savefig("pred_test_onefigure_pure_vs_latent_vs_anchor_residual.pdf", bbox_inches="tight")
fig.savefig("pred_test_onefigure_pure_vs_latent_vs_anchor_residual.png", dpi=300, bbox_inches="tight")


In [ ]:
# Export daily predictions for decision-oriented analysis

import numpy as np
import pandas as pd
import torch

def _pick_artifact(run_dicts, artifacts, key="score"):
    vals = np.array([d[key] for d in run_dicts], dtype=float)
    med = np.median(vals)
    idx = int(np.argmin(np.abs(vals - med)))
    seed = run_dicts[idx]["seed"]
    return seed, artifacts[seed]["model"], artifacts[seed]["ema"]

def _predict_for_export(model, ema, X, month):
    return model_predict_real(
        model,
        X,
        month_idx=month,
        batch_size=4096,
        use_ema=True,
        ema=ema,
    ).detach().cpu().numpy().reshape(-1)

def _actual_for_export(y_tensor, model):
    return scaled_to_real_torch(y_tensor, model, to_cpu=True).detach().cpu().numpy().reshape(-1)

pure_seed, pure_model, pure_ema = _pick_artifact(pure_runs, pure_artifacts)
hyb_seed, hyb_model, hyb_ema = _pick_artifact(hyb_runs, hyb_artifacts)
moe_seed, moe_model, moe_ema = _pick_artifact(moe_runs, moe_artifacts)
lat_seed, lat_model, lat_ema = _pick_artifact(lat_runs, lat_artifacts)
wc_seed, wc_model, wc_ema = _pick_artifact(wc_runs, wc_artifacts)

def build_split_export(split_name, dates_split, X_split, y_split, month_split):
    actual = _actual_for_export(y_split, pure_model)

    out = pd.DataFrame({
        "date": pd.to_datetime(dates_split),
        "split": split_name,
        "actual": actual,
    })

    out["Pure_CNN_BiLSTM_Attn"] = _predict_for_export(pure_model, pure_ema, X_split, month_split)
    out["Hybrid_Anchor_Residual"] = _predict_for_export(hyb_model, hyb_ema, X_split, month_split)
    out["MoE_Gated_Hybrid"] = _predict_for_export(moe_model, moe_ema, X_split, month_split)
    out["Latent_Physical_Regularization"] = _predict_for_export(lat_model, lat_ema, X_split, month_split)
    out["WeatherCalib_PreNet"] = _predict_for_export(wc_model, wc_ema, X_split, month_split)

    out["PHYS_only"] = predict_y_phys_from_hybrid(
        hyb_model,
        X_split,
        month_split,
        batch_size=4096,
        use_ema=True,
        ema=hyb_ema,
    ).detach().cpu().numpy().reshape(-1)

    out["Persistence_D1"] = out["actual"].shift(1)
    out["Persistence_D1"] = out["Persistence_D1"].fillna(out["actual"].iloc[0])

    return out

df_main_export = build_split_export(
    "MAIN_TEST",
    dates_main_test,
    X_main_test_tensor,
    y_main_test_tensor,
    month_main_test_tensor,
)

df_pred_export = build_split_export(
    "PRED_TEST",
    dates_pred_test,
    X_pred_test_tensor,
    y_pred_test_true_tensor,
    month_pred_test_tensor,
)

df_export = pd.concat([df_main_export, df_pred_export], ignore_index=True)

seed_info = {
    "Pure_CNN_BiLSTM_Attn_seed": pure_seed,
    "Hybrid_Anchor_Residual_seed": hyb_seed,
    "MoE_Gated_Hybrid_seed": moe_seed,
    "Latent_Physical_Regularization_seed": lat_seed,
    "WeatherCalib_PreNet_seed": wc_seed,
}
print("Representative seeds:", seed_info)

df_export.to_csv("physics_injection_daily_predictions_for_decision.csv", index=False)
pd.DataFrame([seed_info]).to_csv("physics_injection_representative_seeds.csv", index=False)

print("Saved: physics_injection_daily_predictions_for_decision.csv")
print("Saved: physics_injection_representative_seeds.csv")
display(df_export.head())
display(df_export.tail())
